In [1]:
!pip install numpy scipy pandas plotly streamlit networkx pulp -q
!pip install stable_baselines3 gymnasium -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 10.0 MB/s eta 0:00:00


In [2]:
import numpy as np
from scipy.optimize import linprog, minimize
import itertools
import warnings
warnings.filterwarnings('ignore')

# ── Dominance Elimination ─────────────────────────────────────────────────────
def dominance_elimination(A, B, max_rounds=20):
    rows = list(range(A.shape[0]))
    cols = list(range(A.shape[1]))
    changed = True
    rounds  = 0
    log     = []
    while changed and rounds < max_rounds:
        changed = False
        rounds += 1
        to_remove = []
        for r in rows:
            for r2 in rows:
                if r == r2: continue
                if all(A[r2, c] > A[r, c] for c in cols):
                    to_remove.append(r)
                    log.append(f"Row {r} strictly dominated by Row {r2}")
                    break
        for r in set(to_remove):
            if r in rows:
                rows.remove(r); changed = True
        to_remove = []
        for c in cols:
            for c2 in cols:
                if c == c2: continue
                if all(B[r, c2] > B[r, c] for r in rows):
                    to_remove.append(c)
                    log.append(f"Col {c} strictly dominated by Col {c2}")
                    break
        for c in set(to_remove):
            if c in cols:
                cols.remove(c); changed = True
    return rows, cols, log

# ── Pure Nash ─────────────────────────────────────────────────────────────────
def find_pure_nash(A, B):
    n_rows, n_cols = A.shape
    equilibria = []
    for r in range(n_rows):
        for c in range(n_cols):
            if A[r,c] == np.max(A[:,c]) and B[r,c] == np.max(B[r,:]):
                equilibria.append((r, c, A[r,c], B[r,c]))
    return equilibria

# ── Mixed Nash 2×2 ────────────────────────────────────────────────────────────
def find_mixed_nash_2x2(A, B):
    results = {}
    denom_p = B[0,0] - B[1,0] - B[0,1] + B[1,1]
    denom_q = A[0,0] - A[0,1] - A[1,0] + A[1,1]
    if abs(denom_p) > 1e-10:
        p = (B[1,1] - B[1,0]) / denom_p
        if 0 <= p <= 1:
            results['p']          = round(p, 4)
            results['row_payoff'] = round(p*A[0,0]+(1-p)*A[1,0], 4)
    if abs(denom_q) > 1e-10:
        q = (A[1,1] - A[0,1]) / denom_q
        if 0 <= q <= 1:
            results['q']          = round(q, 4)
            results['col_payoff'] = round(q*B[0,0]+(1-q)*B[0,1], 4)
    return results

# ── Correlated Equilibrium via LP ─────────────────────────────────────────────
def find_correlated_equilibrium(A, B):
    """
    Maximize social welfare sum(p_ij * (A_ij + B_ij))
    subject to incentive compatibility constraints.
    For each player i and each pair of strategies (a, a'):
    sum_s p(a,s) * [u_i(a,s) - u_i(a',s)] >= 0
    plus p >= 0 and sum(p) = 1.
    """
    n, m = A.shape
    n_vars = n * m

    # Objective: maximize social welfare → minimize negative
    c_obj = -(A + B).flatten()

    # Constraints
    A_ub = []
    b_ub = []

    # Row player IC: for each r, r2 pair
    for r in range(n):
        for r2 in range(n):
            if r == r2: continue
            row = np.zeros(n_vars)
            for col in range(m):
                row[r*m + col] = -(A[r, col] - A[r2, col])
            A_ub.append(row)
            b_ub.append(0)

    # Col player IC: for each c, c2 pair
    for c in range(m):
        for c2 in range(m):
            if c == c2: continue
            row = np.zeros(n_vars)
            for r in range(n):
                row[r*m + c] = -(B[r, c] - B[r, c2])
            A_ub.append(row)
            b_ub.append(0)

    # Equality: probabilities sum to 1
    A_eq = np.ones((1, n_vars))
    b_eq = [1.0]

    bounds = [(0, None)] * n_vars

    result = linprog(
        c_obj,
        A_ub=np.array(A_ub), b_ub=np.array(b_ub),
        A_eq=A_eq, b_eq=b_eq,
        bounds=bounds, method='highs'
    )

    if result.success:
        p = result.x.reshape(n, m)
        sw = np.sum(p * (A + B))
        return p, sw
    return None, None

# ── Full Solver ───────────────────────────────────────────────────────────────
def solve_game(A, B, labels_row=None, labels_col=None):
    n, m = A.shape
    if labels_row is None: labels_row = [f"R{i}" for i in range(n)]
    if labels_col is None: labels_col = [f"C{j}" for j in range(m)]

    print("=" * 60)
    print("GAME THEORY SOLVER")
    print("=" * 60)
    print(f"Game: {n}×{m}")
    print(f"\nRow payoff matrix A:\n{A}")
    print(f"\nCol payoff matrix B:\n{B}")

    # Step 1: Dominance
    rows, cols, log = dominance_elimination(A, B)
    print(f"\nDominance Elimination:")
    if log:
        for l in log: print(f"  → {l}")
    else:
        print("  No dominated strategies found")
    print(f"  Surviving: Rows {rows}, Cols {cols}")

    # Step 2: Pure Nash
    pure = find_pure_nash(A, B)
    print(f"\nPure Strategy Nash Equilibria: {len(pure)} found")
    for r, c, ar, bc in pure:
        print(f"  ({labels_row[r]}, {labels_col[c]}) → "
              f"Payoffs: Row={ar}, Col={bc}")

    # Step 3: Mixed Nash (2×2 only)
    if n == 2 and m == 2:
        mixed = find_mixed_nash_2x2(A, B)
        print(f"\nMixed Strategy Nash Equilibrium:")
        if 'p' in mixed:
            print(f"  Row player: p={mixed['p']} on {labels_row[0]}, "
                  f"{1-mixed['p']} on {labels_row[1]}")
        if 'q' in mixed:
            print(f"  Col player: q={mixed['q']} on {labels_col[0]}, "
                  f"{1-mixed['q']} on {labels_col[1]}")

    # Step 4: Correlated Equilibrium
    ce, sw = find_correlated_equilibrium(A, B)
    if ce is not None:
        print(f"\nCorrelated Equilibrium (max social welfare):")
        print(f"  Distribution:\n{np.round(ce, 4)}")
        print(f"  Social welfare: {sw:.4f}")
        # Nash social welfare
        nash_sw = max(A[r,c]+B[r,c] for r,c,_,_ in pure) if pure else 0
        print(f"  Nash social welfare: {nash_sw:.4f}")
        if sw > nash_sw + 0.001:
            print(f"  → Correlated equilibrium improves welfare by "
                  f"{sw-nash_sw:.4f} ✓")

    print("=" * 60)
    return {'pure': pure, 'mixed': mixed if n==2 and m==2 else {},
            'correlated': ce, 'social_welfare': sw}


# ── Test on classic games ─────────────────────────────────────────────────────
print("TEST 1: Prisoner's Dilemma")
A_pd = np.array([[-1, -3], [0, -2]], dtype=float)
B_pd = np.array([[-1, 0],  [-3,-2]], dtype=float)
r1 = solve_game(A_pd, B_pd, ['Cooperate','Defect'], ['Cooperate','Defect'])

print("\nTEST 2: Chicken Game")
A_ch = np.array([[0, -1], [1, -10]], dtype=float)
B_ch = np.array([[0, 1], [-1,-10]], dtype=float)
r2 = solve_game(A_ch, B_ch, ['Swerve','Straight'], ['Swerve','Straight'])

print("\nTEST 3: Battle of the Sexes")
A_bs = np.array([[3, 0], [0, 1]], dtype=float)
B_bs = np.array([[1, 0], [0, 3]], dtype=float)
r3 = solve_game(A_bs, B_bs, ['Opera','Football'], ['Opera','Football'])

TEST 1: Prisoner's Dilemma
GAME THEORY SOLVER
Game: 2×2

Row payoff matrix A:
[[-1. -3.]
 [ 0. -2.]]

Col payoff matrix B:
[[-1.  0.]
 [-3. -2.]]

Dominance Elimination:
  → Row 0 strictly dominated by Row 1
  → Col 0 strictly dominated by Col 1
  Surviving: Rows [1], Cols [1]

Pure Strategy Nash Equilibria: 1 found
  (Defect, Defect) → Payoffs: Row=-2.0, Col=-2.0

Mixed Strategy Nash Equilibrium:

Correlated Equilibrium (max social welfare):
  Distribution:
[[-0.  0.]
 [-0.  1.]]
  Social welfare: -4.0000
  Nash social welfare: -4.0000

TEST 2: Chicken Game
GAME THEORY SOLVER
Game: 2×2

Row payoff matrix A:
[[  0.  -1.]
 [  1. -10.]]

Col payoff matrix B:
[[  0.   1.]
 [ -1. -10.]]

Dominance Elimination:
  No dominated strategies found
  Surviving: Rows [0, 1], Cols [0, 1]

Pure Strategy Nash Equilibria: 2 found
  (Swerve, Straight) → Payoffs: Row=-1.0, Col=1.0
  (Straight, Swerve) → Payoffs: Row=1.0, Col=-1.0

Mixed Strategy Nash Equilibrium:
  Row player: p=0.9 on Swerve, 0.0999999

In [3]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

# ── Prisoner's Dilemma: Iterated Play ────────────────────────────────────────
STRATEGIES = {
    'Always Defect':    lambda h: 'D',
    'Always Cooperate': lambda h: 'C',
    'Tit-for-Tat':      lambda h: h[-1] if h else 'C',
    'Grim Trigger':     lambda h: 'D' if 'D' in h else 'C',
    'Random':           lambda h: np.random.choice(['C','D']),
}

def run_ipd(strat1_name, strat2_name, rounds=50):
    s1 = STRATEGIES[strat1_name]
    s2 = STRATEGIES[strat2_name]
    h1, h2 = [], []
    scores1, scores2 = [], []
    PAYOFF = {('C','C'): (-1,-1), ('C','D'): (-3,0),
              ('D','C'): (0,-3),  ('D','D'): (-2,-2)}
    for _ in range(rounds):
        a1 = s1(h2); a2 = s2(h1)
        p1, p2 = PAYOFF[(a1, a2)]
        h1.append(a1); h2.append(a2)
        scores1.append(p1); scores2.append(p2)
    return h1, h2, scores1, scores2

print("=" * 55)
print("PRISONER'S DILEMMA — ITERATED PLAY TOURNAMENT")
print("=" * 55)
strat_names = list(STRATEGIES.keys())
tournament  = np.zeros((len(strat_names), len(strat_names)))
for i, s1 in enumerate(strat_names):
    for j, s2 in enumerate(strat_names):
        _, _, sc1, _ = run_ipd(s1, s2, rounds=100)
        tournament[i, j] = sum(sc1)
print(f"\nTournament payoff matrix (row player total score):")
df_t = pd.DataFrame(tournament,
                    index=strat_names, columns=strat_names) if True else None
df_t = pd.DataFrame(tournament.round(0).astype(int),
                    index=strat_names, columns=strat_names)
print(df_t.to_string())
print(f"\nOverall winner: "
      f"{strat_names[np.argmax(tournament.sum(axis=1))]}")

# ── Cournot Oligopoly ─────────────────────────────────────────────────────────
def cournot_nash(a=100, b=1, c1=10, c2=10):
    """
    Inverse demand: P = a - b*(q1+q2)
    Firm i profit: pi_i = (P - ci)*qi
    Best response: q1* = (a-c1)/2b - q2/2
    Nash: q1 = q2 = (a-c)/3b (symmetric case)
    """
    q1_nash = (a - 2*c1 + c2) / (3*b)
    q2_nash = (a - 2*c2 + c1) / (3*b)
    Q_nash  = q1_nash + q2_nash
    P_nash  = a - b * Q_nash
    pi1_nash = (P_nash - c1) * q1_nash
    pi2_nash = (P_nash - c2) * q2_nash

    Q_monopoly  = (a - c1) / (2*b)
    P_monopoly  = a - b * Q_monopoly
    pi_monopoly = (P_monopoly - c1) * Q_monopoly

    Q_competitive = (a - c1) / b
    P_competitive = c1

    return {
        'q1_nash': round(q1_nash, 2),
        'q2_nash': round(q2_nash, 2),
        'Q_nash':  round(Q_nash, 2),
        'P_nash':  round(P_nash, 2),
        'pi1_nash':round(pi1_nash, 2),
        'pi2_nash':round(pi2_nash, 2),
        'Q_monopoly':  round(Q_monopoly, 2),
        'P_monopoly':  round(P_monopoly, 2),
        'pi_monopoly': round(pi_monopoly, 2),
        'Q_competitive': round(Q_competitive, 2),
        'P_competitive': round(P_competitive, 2),
    }

def best_response(q_other, firm=1, a=100, b=1, c1=10, c2=10):
    c = c1 if firm == 1 else c2
    return max(0, (a - c - b*q_other) / (2*b))

print("\n" + "=" * 55)
print("COURNOT OLIGOPOLY")
print("=" * 55)
r = cournot_nash()
print(f"Nash Equilibrium:")
print(f"  q1* = {r['q1_nash']}, q2* = {r['q2_nash']}")
print(f"  Market price P* = {r['P_nash']}")
print(f"  Profit: π1={r['pi1_nash']}, π2={r['pi2_nash']}")
print(f"\nMonopoly benchmark:")
print(f"  Q = {r['Q_monopoly']}, P = {r['P_monopoly']}, "
      f"π = {r['pi_monopoly']}")
print(f"\nPerfect competition benchmark:")
print(f"  Q = {r['Q_competitive']}, P = {r['P_competitive']}, π = 0")
print(f"\nNash vs Monopoly: firms produce "
      f"{r['Q_nash']/r['Q_monopoly']:.2f}x monopoly output")
print(f"Nash vs Competition: {r['Q_nash']/r['Q_competitive']:.2f}x "
      f"competitive output")

# Visualize best response curves
q_range = np.linspace(0, 60, 200)
br1 = [best_response(q, firm=1) for q in q_range]
br2 = [best_response(q, firm=2) for q in q_range]

fig_c = go.Figure()
fig_c.add_trace(go.Scatter(x=q_range, y=br1, mode='lines',
    name='BR₁(q₂)', line=dict(color='#60a5fa', width=2.5)))
fig_c.add_trace(go.Scatter(x=br2, y=q_range, mode='lines',
    name='BR₂(q₁)', line=dict(color='#34d399', width=2.5)))
fig_c.add_trace(go.Scatter(
    x=[r['q1_nash']], y=[r['q2_nash']], mode='markers',
    name='Nash Eq', marker=dict(color='#f59e0b', size=14, symbol='star')))
fig_c.update_layout(
    title="Cournot Oligopoly — Best Response Curves",
    xaxis_title="q₁", yaxis_title="q₂",
    template="plotly_dark", height=450,
    paper_bgcolor="#0e1117", plot_bgcolor="#0e1117",
    font=dict(color='white'))
fig_c.show()

# ── Public Goods Game ─────────────────────────────────────────────────────────
def public_goods_simulation(n_players=5, endowment=10,
                             return_rate=1.6, rounds=30, noise=0.1):
    """
    Each player i contributes c_i ∈ [0, endowment].
    Total contribution multiplied by return_rate and shared equally.
    Nash: contribute 0 if return_rate < n_players (usually).
    Socially optimal: contribute everything if return_rate > 1.
    """
    contributions = np.ones(n_players) * endowment * 0.5
    history = []

    for round_n in range(rounds):
        total  = np.sum(contributions)
        share  = return_rate * total / n_players
        payoff = endowment - contributions + share

        history.append({
            'round': round_n,
            'avg_contribution': np.mean(contributions),
            'total_contribution': total,
            'avg_payoff': np.mean(payoff)
        })

        # Imitation + noise: players copy high-payoff strategies
        best_player = np.argmax(payoff)
        new_contribs = []
        for i in range(n_players):
            if payoff[i] < np.mean(payoff):
                new_c = contributions[best_player] + np.random.normal(0, noise)
            else:
                new_c = contributions[i] + np.random.normal(0, noise * 0.5)
            new_contribs.append(np.clip(new_c, 0, endowment))
        contributions = np.array(new_contribs)

    return pd.DataFrame(history)

print("\n" + "=" * 55)
print("PUBLIC GOODS GAME")
print("=" * 55)
pg_df = public_goods_simulation(n_players=5, return_rate=1.6, rounds=40)
print(f"Round 1  avg contribution: "
      f"{pg_df.iloc[0]['avg_contribution']:.2f}")
print(f"Round 20 avg contribution: "
      f"{pg_df.iloc[20]['avg_contribution']:.2f}")
print(f"Round 40 avg contribution: "
      f"{pg_df.iloc[-1]['avg_contribution']:.2f}")
print(f"Nash prediction: 0.0 (free rider)")
print(f"Social optimum: 10.0 (full contribution)")
print(f"Observed: slow drift toward free-riding ✓")

print("\nAll classic games complete ✓")

PRISONER'S DILEMMA — ITERATED PLAY TOURNAMENT

Tournament payoff matrix (row player total score):
                  Always Defect  Always Cooperate  Tit-for-Tat  Grim Trigger  Random
Always Defect              -200                 0         -198          -198     -96
Always Cooperate           -300              -100         -100          -100    -200
Tit-for-Tat                -201              -100         -100          -100    -151
Grim Trigger               -201              -100         -100          -100    -114
Random                     -255               -48         -148          -259    -136

Overall winner: Grim Trigger

COURNOT OLIGOPOLY
Nash Equilibrium:
  q1* = 30.0, q2* = 30.0
  Market price P* = 40.0
  Profit: π1=900.0, π2=900.0

Monopoly benchmark:
  Q = 45.0, P = 55.0, π = 2025.0

Perfect competition benchmark:
  Q = 90.0, P = 10, π = 0

Nash vs Monopoly: firms produce 1.33x monopoly output
Nash vs Competition: 0.67x competitive output



PUBLIC GOODS GAME
Round 1  avg contribution: 5.00
Round 20 avg contribution: 2.88
Round 40 avg contribution: 2.07
Nash prediction: 0.0 (free rider)
Social optimum: 10.0 (full contribution)
Observed: slow drift toward free-riding ✓

All classic games complete ✓


In [4]:
from scipy.integrate import solve_ivp
import numpy as np
import plotly.graph_objects as go

# ── Replicator Dynamics ───────────────────────────────────────────────────────
def replicator_rhs(t, x, A):
    """
    dx_i/dt = x_i * (f_i(x) - f_bar(x))
    f_i(x)    = (A @ x)[i]    — fitness of strategy i
    f_bar(x)  = x @ A @ x     — average fitness
    """
    f      = A @ x
    f_bar  = x @ f
    dxdt   = x * (f - f_bar)
    return dxdt

def run_replicator(A, x0, t_max=20, n_points=1000):
    """Solve replicator ODE for payoff matrix A from initial state x0."""
    t_span = (0, t_max)
    t_eval = np.linspace(0, t_max, n_points)
    sol = solve_ivp(replicator_rhs, t_span, x0, t_eval=t_eval,
                    args=(A,), method='RK45', rtol=1e-8)
    return sol.t, sol.y.T  # shape (n_points, n_strategies)

# ── Classic Evolutionary Games ────────────────────────────────────────────────
EVOLUTIONARY_GAMES = {
    'Prisoner\'s Dilemma': {
        'A': np.array([[-1., -3.], [0., -2.]]),
        'strategies': ['Cooperate', 'Defect'],
        'prediction': 'All Defect (all-D is ESS)'
    },
    'Hawk-Dove': {
        'A': np.array([[0., 3.], [1., 2.]]),
        'strategies': ['Hawk', 'Dove'],
        'prediction': 'Mixed equilibrium (both strategies coexist)'
    },
    'Stag Hunt': {
        'A': np.array([[4., 0.], [3., 3.]]),
        'strategies': ['Stag', 'Hare'],
        'prediction': 'Two stable equilibria (all-Stag or all-Hare)'
    },
    'Rock-Paper-Scissors': {
        'A': np.array([[0., -1., 1.], [1., 0., -1.], [-1., 1., 0.]]),
        'strategies': ['Rock', 'Paper', 'Scissors'],
        'prediction': 'Neutrally stable cycles (no convergence)'
    }
}

print("=" * 60)
print("EVOLUTIONARY GAME THEORY — REPLICATOR DYNAMICS")
print("=" * 60)

for name, game in EVOLUTIONARY_GAMES.items():
    A   = game['A']
    n_s = len(game['strategies'])
    x0  = np.ones(n_s) / n_s  # start at equal mix

    t, X = run_replicator(A, x0, t_max=15)

    print(f"\n{name}")
    print(f"  Prediction: {game['prediction']}")
    print(f"  Initial state: {x0.round(3)}")
    print(f"  Final state:   {X[-1].round(4)}")

    fig = go.Figure()
    colors = ['#60a5fa','#34d399','#f59e0b','#f87171']
    for i, strat in enumerate(game['strategies']):
        fig.add_trace(go.Scatter(
            x=t, y=X[:,i], mode='lines',
            name=strat, line=dict(color=colors[i], width=2.5)
        ))
    fig.update_layout(
        title=f"Replicator Dynamics: {name}",
        xaxis_title="Time", yaxis_title="Population Share",
        template="plotly_dark", height=380,
        paper_bgcolor="#0e1117", plot_bgcolor="#0e1117",
        font=dict(color='white'),
        legend=dict(orientation="h", y=1.1)
    )
    fig.show()

# ── Simplex Phase Portrait (3-strategy games) ─────────────────────────────────
def barycentric_to_cartesian(x):
    """Convert 3-strategy simplex coords to 2D Cartesian."""
    v = np.array([[0.0, 0.0], [1.0, 0.0], [0.5, np.sqrt(3)/2]])
    return x[0]*v[0] + x[1]*v[1] + x[2]*v[2]

def plot_simplex_portrait(A, strategies, title, n_starts=15):
    """Phase portrait on the strategy simplex for 3-strategy games."""
    fig = go.Figure()

    # Draw simplex boundary
    corners = np.array([[1,0,0],[0,1,0],[0,0,1],[1,0,0]], dtype=float)
    cart    = np.array([barycentric_to_cartesian(c) for c in corners])
    fig.add_trace(go.Scatter(
        x=cart[:,0], y=cart[:,1], mode='lines',
        line=dict(color='#374151', width=2), showlegend=False
    ))

    # Corner labels
    for i, (name, corner) in enumerate(zip(strategies,
                                            [[1,0,0],[0,1,0],[0,0,1]])):
        pos = barycentric_to_cartesian(np.array(corner, dtype=float))
        fig.add_annotation(x=pos[0], y=pos[1]+0.04, text=name,
                           font=dict(color='white', size=12),
                           showarrow=False)

    # Trajectories from random starting points
    np.random.seed(42)
    for _ in range(n_starts):
        raw = np.random.dirichlet(np.ones(3))
        t, X = run_replicator(A, raw, t_max=10, n_points=300)
        cart_traj = np.array([barycentric_to_cartesian(X[i])
                               for i in range(len(t))])
        fig.add_trace(go.Scatter(
            x=cart_traj[:,0], y=cart_traj[:,1], mode='lines',
            line=dict(color='rgba(96,165,250,0.4)', width=1.2),
            showlegend=False
        ))
        # Arrow at midpoint
        mid = len(t)//2
        fig.add_annotation(
            x=cart_traj[mid+1,0], y=cart_traj[mid+1,1],
            ax=cart_traj[mid,0],  ay=cart_traj[mid,1],
            xref='x', yref='y', axref='x', ayref='y',
            showarrow=True, arrowhead=2,
            arrowcolor='rgba(96,165,250,0.6)', arrowsize=1
        )

    # Nash equilibrium center
    center = barycentric_to_cartesian(np.array([1/3, 1/3, 1/3]))
    fig.add_trace(go.Scatter(
        x=[center[0]], y=[center[1]], mode='markers',
        name='Nash (1/3,1/3,1/3)',
        marker=dict(color='#f59e0b', size=12, symbol='star')
    ))

    fig.update_layout(
        title=title,
        template="plotly_dark", height=500,
        paper_bgcolor="#0e1117", plot_bgcolor="#0e1117",
        font=dict(color='white'),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False,
                   scaleanchor="x")
    )
    return fig

A_rps = EVOLUTIONARY_GAMES['Rock-Paper-Scissors']['A']
fig_rps = plot_simplex_portrait(
    A_rps, ['Rock','Paper','Scissors'],
    "Rock-Paper-Scissors: Replicator Dynamics on Simplex"
)
fig_rps.show()

A_sh = EVOLUTIONARY_GAMES['Stag Hunt']['A']
# Embed in 3-strategy simplex with dummy 3rd strategy
A_sh3 = np.array([[4.,0.,0.],[3.,3.,0.],[0.,0.,0.]])
fig_sh = plot_simplex_portrait(
    A_sh3, ['Stag','Hare','—'],
    "Stag Hunt: Two Stable Equilibria"
)
fig_sh.show()

print("\nEvolutionary dynamics complete ✓")

EVOLUTIONARY GAME THEORY — REPLICATOR DYNAMICS

Prisoner's Dilemma
  Prediction: All Defect (all-D is ESS)
  Initial state: [0.5 0.5]
  Final state:   [0.     0.9997]



Hawk-Dove
  Prediction: Mixed equilibrium (both strategies coexist)
  Initial state: [0.5 0.5]
  Final state:   [0.5 0.5]



Stag Hunt
  Prediction: Two stable equilibria (all-Stag or all-Hare)
  Initial state: [0.5 0.5]
  Final state:   [0. 1.]



Rock-Paper-Scissors
  Prediction: Neutrally stable cycles (no convergence)
  Initial state: [0.333 0.333 0.333]
  Final state:   [0.3333 0.3333 0.3333]



Evolutionary dynamics complete ✓


In [5]:
import numpy as np
import itertools
import pandas as pd
import plotly.graph_objects as go
from math import factorial

# ── Exact Shapley Value ───────────────────────────────────────────────────────
def shapley_value(v, n):
    """
    Exact Shapley value computation.
    v: characteristic function v(S) → value of coalition S
       S represented as frozenset of player indices
    n: number of players
    Returns array of Shapley values φ_i for each player i.
    """
    players = list(range(n))
    phi     = np.zeros(n)

    for i in players:
        others  = [p for p in players if p != i]
        for r in range(len(others) + 1):
            for S_list in itertools.combinations(others, r):
                S      = frozenset(S_list)
                S_with = frozenset(S_list + (i,))
                # Marginal contribution of i to coalition S
                marginal = v(S_with) - v(S)
                # Weight: |S|!(n-|S|-1)!/n!
                weight = (factorial(len(S)) *
                          factorial(n - len(S) - 1) /
                          factorial(n))
                phi[i] += weight * marginal

    return phi


def monte_carlo_shapley(v, n, n_samples=10000):
    """
    Monte Carlo approximation for large n.
    Randomly permute players and compute marginal contributions.
    """
    players = list(range(n))
    phi     = np.zeros(n)

    for _ in range(n_samples):
        perm = np.random.permutation(players)
        coalition = frozenset()
        for i in perm:
            coalition_with = coalition | {i}
            phi[i] += v(coalition_with) - v(coalition)
            coalition = coalition_with

    return phi / n_samples


# ── Economic Examples ─────────────────────────────────────────────────────────
print("=" * 60)
print("SHAPLEY VALUE CALCULATOR")
print("=" * 60)

# Example 1: Simple production game
# Three workers with complementary skills
# Value generated depends on combination
def production_game(S):
    """
    Worker 0: programmer, Worker 1: designer, Worker 2: manager
    v({0,1}) = 50 (programmer+designer build product)
    v({0,2}) = 30 (programmer+manager plan)
    v({1,2}) = 20 (designer+manager present)
    v({0,1,2}) = 100 (full team)
    """
    s = set(S)
    if s == {0,1,2}: return 100
    if s == {0,1}:   return 50
    if s == {0,2}:   return 30
    if s == {1,2}:   return 20
    if s == {0}:     return 10
    if s == {1}:     return 5
    if s == {2}:     return 8
    return 0

phi = shapley_value(production_game, 3)
print(f"\nExample 1: Production Game")
print(f"  Total value: {production_game(frozenset([0,1,2]))}")
print(f"  Shapley values:")
names = ['Programmer','Designer','Manager']
for i, (name, p) in enumerate(zip(names, phi)):
    print(f"    {name}: φ={p:.2f} ({p/sum(phi)*100:.1f}%)")
print(f"  Sum of Shapley values = {sum(phi):.2f} "
      f"(= total value ✓ efficiency axiom)")

# Example 2: Voting game
def voting_game(S):
    """Weighted majority: Player 0 has 3 votes, 1 has 2, 2 has 1.
    Win requires 4+ votes."""
    weights = {0: 3, 1: 2, 2: 1}
    total = sum(weights[i] for i in S)
    return 1 if total >= 4 else 0

phi_v = shapley_value(voting_game, 3)
print(f"\nExample 2: Weighted Voting Game (3,2,1 votes; quota=4)")
print(f"  Shapley-Shubik power indices:")
for i, p in enumerate(phi_v):
    print(f"    Player {i} ({[3,2,1][i]} votes): {p:.4f} ({p*100:.1f}%)")

# ── SHAP Connection ───────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("SHAP CONNECTION: Shapley Values in Machine Learning")
print(f"{'='*60}")

# Train a small linear model and compute SHAP values two ways:
# 1. Game-theoretic Shapley value treating features as players
# 2. Show they're identical

from sklearn.linear_model import LinearRegression
import numpy as np

np.random.seed(42)
n_samples = 200
X = np.random.randn(n_samples, 3)
# True relationship: Y = 2*X0 + 5*X1 - 1*X2 + noise
y = 2*X[:,0] + 5*X[:,1] - 1*X[:,2] + np.random.randn(n_samples)*0.5

model = LinearRegression().fit(X, y)
print(f"\nModel: Y = {model.coef_[0]:.2f}*X₀ + "
      f"{model.coef_[1]:.2f}*X₁ + {model.coef_[2]:.2f}*X₂")
print(f"True:  Y = 2.00*X₀ + 5.00*X₁ + (-1.00)*X₂")

# Compute Shapley values for a single prediction
x_explain = X[0]
baseline  = y.mean()

def model_value(S, x=x_explain, X_bg=X, model=model):
    """
    Value function for SHAP: E[model(x) | x_S = x_S]
    Features in S are observed; others integrated over background.
    """
    if len(S) == 0:
        return baseline
    S = list(S)
    X_perm = X_bg.copy()
    for i in S:
        X_perm[:, i] = x[i]
    return model.predict(X_perm).mean()

phi_ml = shapley_value(model_value, 3)
print(f"\nSHAP values for observation x₀:")
print(f"  Model prediction: {model.predict([x_explain])[0]:.4f}")
print(f"  Baseline (mean):  {baseline:.4f}")
feat_names = ['X₀ (coef≈2)', 'X₁ (coef≈5)', 'X₂ (coef≈-1)']
for i, (name, p) in enumerate(zip(feat_names, phi_ml)):
    print(f"  {name}: SHAP={p:+.4f}")
print(f"  Sum of SHAP + baseline = "
      f"{sum(phi_ml)+baseline:.4f} = prediction ✓")
print(f"\n  Key insight: SHAP values are Shapley values from")
print(f"  cooperative game theory applied to ML features.")
print(f"  Same mathematical object, different domain.")
print(f"  This is why SHAP satisfies efficiency, symmetry,")
print(f"  dummy, and additivity — the four Shapley axioms.")

# ── Waterfall visualization ───────────────────────────────────────────────────
fig_shap = go.Figure(go.Waterfall(
    name="SHAP",
    orientation="v",
    measure=["absolute"] + ["relative"]*3 + ["total"],
    x=["Baseline", "X₀ (prog.)", "X₁ (design.)", "X₂ (mgr.)", "Prediction"],
    y=[baseline, phi_ml[0], phi_ml[1], phi_ml[2], 0],
    connector=dict(line=dict(color='#374151')),
    increasing=dict(marker=dict(color='#34d399')),
    decreasing=dict(marker=dict(color='#f87171')),
    totals=dict(marker=dict(color='#60a5fa'))
))
fig_shap.update_layout(
    title="SHAP Waterfall: Shapley Values for ML Feature Attribution",
    template="plotly_dark", height=420,
    paper_bgcolor="#0e1117", plot_bgcolor="#0e1117",
    font=dict(color='white'), yaxis_title="Prediction Value"
)
fig_shap.show()

print("\nShapley values complete ✓")

SHAPLEY VALUE CALCULATOR

Example 1: Production Game
  Total value: 100
  Shapley values:
    Programmer: φ=41.17 (41.2%)
    Designer: φ=33.67 (33.7%)
    Manager: φ=25.17 (25.2%)
  Sum of Shapley values = 100.00 (= total value ✓ efficiency axiom)

Example 2: Weighted Voting Game (3,2,1 votes; quota=4)
  Shapley-Shubik power indices:
    Player 0 (3 votes): 0.6667 (66.7%)
    Player 1 (2 votes): 0.1667 (16.7%)
    Player 2 (1 votes): 0.1667 (16.7%)

SHAP CONNECTION: Shapley Values in Machine Learning

Model: Y = 1.99*X₀ + 5.05*X₁ + -0.94*X₂
True:  Y = 2.00*X₀ + 5.00*X₁ + (-1.00)*X₂

SHAP values for observation x₀:
  Model prediction: -0.3101
  Baseline (mean):  -0.6489
  X₀ (coef≈2): SHAP=+0.8633
  X₁ (coef≈5): SHAP=+0.0436
  X₂ (coef≈-1): SHAP=-0.5680
  Sum of SHAP + baseline = -0.3101 = prediction ✓

  Key insight: SHAP values are Shapley values from
  cooperative game theory applied to ML features.
  Same mathematical object, different domain.
  This is why SHAP satisfies efficienc


Shapley values complete ✓


In [6]:
import numpy as np
import networkx as nx
import plotly.graph_objects as go
import pandas as pd

# ── Network Public Goods Game ─────────────────────────────────────────────────
def network_public_goods(G, endowment=10, return_rate=1.6,
                          rounds=30, seed=42):
    """
    Each node decides contribution to local public good.
    Payoff depends on neighbors' contributions (not global).
    """
    np.random.seed(seed)
    n   = len(G.nodes)
    pos = nx.spring_layout(G, seed=42)

    contributions = np.random.uniform(3, 7, n)
    history       = []

    for round_n in range(rounds):
        payoffs = np.zeros(n)
        for i in G.nodes:
            neighbors = list(G.neighbors(i))
            if not neighbors:
                payoffs[i] = endowment - contributions[i]
                continue
            local_pool = contributions[i] + sum(
                contributions[j] for j in neighbors
            )
            share = return_rate * local_pool / (len(neighbors) + 1)
            payoffs[i] = endowment - contributions[i] + share

        history.append({
            'round': round_n,
            'avg_contribution': np.mean(contributions),
            'gini': np.std(contributions) / (np.mean(contributions)+1e-9)
        })

        # Best response update with noise
        new_c = contributions.copy()
        for i in G.nodes:
            neighbors = list(G.neighbors(i))
            if neighbors:
                best_neighbor = neighbors[np.argmax(
                    [payoffs[j] for j in neighbors]
                )]
                if payoffs[best_neighbor] > payoffs[i]:
                    new_c[i] = (contributions[i] * 0.7 +
                                contributions[best_neighbor] * 0.3 +
                                np.random.normal(0, 0.5))
            new_c[i] = np.clip(new_c[i], 0, endowment)
        contributions = new_c

    return contributions, payoffs, history, pos

# ── Network topologies ────────────────────────────────────────────────────────
NETWORKS = {
    'Complete Graph':   nx.complete_graph(12),
    'Star Network':     nx.star_graph(11),
    'Random (ER)':      nx.erdos_renyi_graph(12, 0.4, seed=42),
    'Scale-Free (BA)':  nx.barabasi_albert_graph(12, 2, seed=42),
    'Small World (WS)': nx.watts_strogatz_graph(12, 4, 0.3, seed=42),
}

print("=" * 60)
print("NETWORK GAMES — PUBLIC GOODS ON GRAPHS")
print("=" * 60)

results = {}
for name, G in NETWORKS.items():
    contribs, payoffs, hist, pos = network_public_goods(G, rounds=40)
    results[name] = {
        'final_contribution': np.mean(contribs),
        'final_payoff': np.mean(payoffs),
        'gini': np.std(contribs)/(np.mean(contribs)+1e-9),
        'density': nx.density(G),
        'avg_clustering': nx.average_clustering(G),
        'contributions': contribs,
        'payoffs': payoffs,
        'history': hist,
        'pos': pos,
        'G': G
    }
    print(f"\n{name}:")
    print(f"  Density: {nx.density(G):.3f} | "
          f"Avg clustering: {nx.average_clustering(G):.3f}")
    print(f"  Final avg contribution: {np.mean(contribs):.2f}")
    print(f"  Final avg payoff: {np.mean(payoffs):.2f}")
    print(f"  Contribution inequality (Gini): "
          f"{np.std(contribs)/(np.mean(contribs)+1e-9):.3f}")

# ── Summary comparison ────────────────────────────────────────────────────────
summary_df = pd.DataFrame({
    'Network': list(results.keys()),
    'Density': [results[n]['density'] for n in results],
    'Avg Contribution': [results[n]['final_contribution']
                         for n in results],
    'Avg Payoff': [results[n]['final_payoff'] for n in results],
    'Inequality': [results[n]['gini'] for n in results]
}).round(3)
print(f"\nSummary across network topologies:")
print(summary_df.to_string(index=False))

# ── Visualize network with contribution heatmap ───────────────────────────────
def plot_network_game(G, contributions, pos, title):
    edge_x, edge_y = [], []
    for e in G.edges():
        x0,y0 = pos[e[0]]; x1,y1 = pos[e[1]]
        edge_x += [x0,x1,None]; edge_y += [y0,y1,None]

    node_x = [pos[n][0] for n in G.nodes]
    node_y = [pos[n][1] for n in G.nodes]

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=edge_x, y=edge_y, mode='lines',
        line=dict(color='rgba(148,163,184,0.3)', width=1.5),
        showlegend=False))
    fig.add_trace(go.Scatter(
        x=node_x, y=node_y, mode='markers+text',
        marker=dict(size=20, color=contributions,
                    colorscale='RdYlGn', colorbar=dict(title='Contribution'),
                    cmin=0, cmax=10,
                    line=dict(color='white', width=1.5)),
        text=[f"{c:.1f}" for c in contributions],
        textfont=dict(color='black', size=9),
        showlegend=False
    ))
    fig.update_layout(
        title=title, template="plotly_dark", height=420,
        paper_bgcolor="#0e1117", plot_bgcolor="#0e1117",
        font=dict(color='white'),
        xaxis=dict(showgrid=False,zeroline=False,showticklabels=False),
        yaxis=dict(showgrid=False,zeroline=False,showticklabels=False)
    )
    return fig

for name in ['Complete Graph','Star Network','Scale-Free (BA)']:
    r = results[name]
    fig = plot_network_game(r['G'], r['contributions'],
                             r['pos'], f"{name} — Final Contributions")
    fig.show()

print("\nNetwork games complete ✓")

NETWORK GAMES — PUBLIC GOODS ON GRAPHS

Complete Graph:
  Density: 1.000 | Avg clustering: 1.000
  Final avg contribution: 0.00
  Final avg payoff: 10.00
  Contribution inequality (Gini): 0.000

Star Network:
  Density: 0.167 | Avg clustering: 0.000
  Final avg contribution: 0.68
  Final avg payoff: 10.21
  Contribution inequality (Gini): 0.784

Random (ER):
  Density: 0.455 | Avg clustering: 0.490
  Final avg contribution: 0.26
  Final avg payoff: 10.08
  Contribution inequality (Gini): 1.253

Scale-Free (BA):
  Density: 0.303 | Avg clustering: 0.351
  Final avg contribution: 0.21
  Final avg payoff: 10.07
  Contribution inequality (Gini): 1.465

Small World (WS):
  Density: 0.364 | Avg clustering: 0.350
  Final avg contribution: 0.67
  Final avg payoff: 10.31
  Contribution inequality (Gini): 0.831

Summary across network topologies:
         Network  Density  Avg Contribution  Avg Payoff  Inequality
  Complete Graph    1.000             0.000      10.000       0.000
    Star Network


Network games complete ✓


In [7]:
import numpy as np
from scipy.optimize import fsolve
import plotly.graph_objects as go

# ── Quantal Response Equilibrium ─────────────────────────────────────────────
def logit_qre(A, B, lam, tol=1e-10, max_iter=1000):
    """
    Logit Quantal Response Equilibrium.
    σ_i(a) = exp(λ * EU_i(a, σ_{-i})) / Z_i
    Players are noisy best responders: better responses
    are chosen more often, not exclusively.
    As λ→∞: QRE → Nash. As λ→0: QRE → uniform random.
    """
    n, m  = A.shape
    # Initialize at uniform
    p = np.ones(n) / n  # row player mixed strategy
    q = np.ones(m) / m  # col player mixed strategy

    for _ in range(max_iter):
        # Expected payoffs
        eu_row = A @ q   # shape (n,)
        eu_col = B.T @ p # shape (m,)

        # Logit best response
        p_new = np.exp(lam * eu_row)
        p_new /= p_new.sum()
        q_new = np.exp(lam * eu_col)
        q_new /= q_new.sum()

        if (np.max(np.abs(p_new - p)) < tol and
            np.max(np.abs(q_new - q)) < tol):
            break

        p, q = p_new, q_new

    return p, q

print("=" * 60)
print("QUANTAL RESPONSE EQUILIBRIUM")
print("=" * 60)

# Prisoner's Dilemma
A_pd = np.array([[-1., -3.], [0., -2.]])
B_pd = np.array([[-1., 0.], [-3., -2.]])

print("\nPrisoner's Dilemma: QRE across precision λ")
print(f"{'λ':>8} {'P(Cooperate)':>14} {'Nash Pred':>12}")
print("-" * 40)
lambdas = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 50.0]
for lam in lambdas:
    p, q = logit_qre(A_pd, B_pd, lam)
    print(f"{lam:>8.2f} {p[0]:>14.4f} {'0.0 (all-D)':>12}")

print(f"\n  λ→0: players choose randomly (≈50% cooperate)")
print(f"  λ→∞: players best-respond perfectly (0% cooperate = Nash)")
print(f"  QRE bridges rationality and bounded rationality ✓")

# Chicken Game — QRE vs Nash comparison
A_ch = np.array([[0., -1.], [1., -10.]])
B_ch = np.array([[0., 1.], [-1., -10.]])

print(f"\nChicken Game: QRE vs Nash")
p_nash = find_mixed_nash_2x2(A_ch, B_ch)
print(f"  Nash mixed: P(Swerve) = {p_nash.get('p', 'N/A')}")
for lam in [0.1, 1.0, 5.0, 20.0]:
    p_qre, _ = logit_qre(A_ch, B_ch, lam)
    print(f"  QRE λ={lam:5.1f}: P(Swerve)={p_qre[0]:.4f}")

# ── λ sweep visualization ─────────────────────────────────────────────────────
lam_range  = np.logspace(-2, 2, 100)
p_coop_pd  = []
p_coop_ch  = []

for lam in lam_range:
    p_pd, _ = logit_qre(A_pd, B_pd, lam)
    p_ch, _ = logit_qre(A_ch, B_ch, lam)
    p_coop_pd.append(p_pd[0])
    p_coop_ch.append(p_ch[0])

fig_qre = go.Figure()
fig_qre.add_trace(go.Scatter(
    x=lam_range, y=p_coop_pd, mode='lines',
    name="PD: P(Cooperate)", line=dict(color='#60a5fa', width=2.5)))
fig_qre.add_trace(go.Scatter(
    x=lam_range, y=p_coop_ch, mode='lines',
    name="Chicken: P(Swerve)", line=dict(color='#34d399', width=2.5)))
fig_qre.add_hline(y=0, line_dash="dot", line_color="#f87171",
                   annotation_text="Nash (PD)",
                   annotation_position="right")
fig_qre.add_hline(y=p_nash.get('p', 0.9), line_dash="dot",
                   line_color="#f59e0b",
                   annotation_text="Nash (Chicken)",
                   annotation_position="right")
fig_qre.update_layout(
    title="QRE: Strategy Probability vs Precision λ<br>"
          "<sub>As λ→∞, QRE converges to Nash equilibrium</sub>",
    xaxis_title="Precision λ (log scale)", xaxis_type="log",
    yaxis_title="Probability of Cooperative Strategy",
    template="plotly_dark", height=420,
    paper_bgcolor="#0e1117", plot_bgcolor="#0e1117",
    font=dict(color='white'), legend=dict(orientation="h", y=1.1)
)
fig_qre.show()

# ── Mechanism Design: VCG Mechanism ──────────────────────────────────────────
print("\n" + "=" * 60)
print("MECHANISM DESIGN: VICKREY-CLARKE-GROVES (VCG)")
print("=" * 60)
print("Objective: Design rules that make truthful reporting")
print("each agent's dominant strategy (incentive compatible).")

def vcg_auction(valuations, costs=None):
    """
    VCG mechanism for a single-item auction.
    Valuations: list of agent valuations v_i for the item.
    Winner: agent with highest valuation (efficient allocation).
    VCG Payment: winner pays second-highest valuation.
    This is the Vickrey second-price auction — truth-telling dominant.
    """
    n = len(valuations)
    valuations = np.array(valuations)

    # Efficient allocation: give item to highest bidder
    winner = np.argmax(valuations)
    allocation = np.zeros(n); allocation[winner] = 1

    # VCG payments: externality imposed on others
    # p_i = max_{j≠i} v_j (what others lose by i getting the item)
    payments = np.zeros(n)
    for i in range(n):
        others = [v for j, v in enumerate(valuations) if j != i]
        # Social welfare without i: best allocation among others
        sw_without_i = max(others)
        # Social welfare with i (excluding i's value): 0 if i wins, else sw
        if allocation[i] == 1:
            payments[i] = sw_without_i
        else:
            payments[i] = 0

    utility = valuations * allocation - payments

    return {
        'winner': winner,
        'allocation': allocation,
        'payments': payments,
        'utility': utility,
        'social_welfare': np.sum(valuations * allocation)
    }

def vcg_public_goods(valuations, cost):
    """
    VCG for binary public good (provide or not).
    Provide if sum(v_i) >= cost.
    VCG payment: Clarke tax ensuring truthfulness.
    """
    n = len(valuations)
    valuations = np.array(valuations, dtype=float)
    total_val  = np.sum(valuations)
    provide    = total_val >= cost

    payments = np.zeros(n)
    for i in range(n):
        others_total = total_val - valuations[i]
        # Would project happen without i?
        without_i = others_total >= cost
        if provide and not without_i:
            # i is pivotal — pays the difference
            payments[i] = cost - others_total
        elif not provide and without_i:
            payments[i] = -(others_total - cost)

    return {
        'provide': provide,
        'payments': payments,
        'total_payment': np.sum(payments),
        'efficiency_gap': cost - total_val if not provide else 0
    }

# Test 1: Auction
print(f"\nTest 1: VCG Auction (second-price)")
vals = [45, 72, 38, 91, 55]
res  = vcg_auction(vals)
print(f"  Valuations: {vals}")
print(f"  Winner: Agent {res['winner']} (valued at {vals[res['winner']]})")
print(f"  Payment: {res['payments'][res['winner']]:.2f} "
      f"(second-price = {sorted(vals)[-2]})")
print(f"  Truth-telling is dominant: bidding true value "
      f"maximizes utility regardless of others ✓")

# Test 2: Incentive compatibility verification
print(f"\nTest 2: Verifying incentive compatibility")
true_val = 91
print(f"  Agent 3 true value: {true_val}")
for reported in [40, 60, 80, 91, 100, 120]:
    fake_vals = vals.copy()
    fake_vals[3] = reported
    r = vcg_auction(fake_vals)
    util = true_val * r['allocation'][3] - r['payments'][3]
    print(f"    Reports {reported:3d}: "
          f"{'wins' if r['allocation'][3] else 'loses'}, "
          f"utility={util:.2f}"
          f"{'  ← TRUTH' if reported==true_val else ''}")

print(f"\n  Truth-telling (report=91) achieves max utility ✓")

# Test 3: Public goods
print(f"\nTest 3: VCG Public Goods Mechanism")
pg_vals = [15, 25, 10, 30, 20]
pg_cost = 80
pg_res  = vcg_public_goods(pg_vals, pg_cost)
print(f"  Valuations: {pg_vals}")
print(f"  Cost: {pg_cost}, Total value: {sum(pg_vals)}")
print(f"  Decision: {'Provide ✓' if pg_res['provide'] else 'Do not provide'}")
print(f"  VCG Payments: {pg_res['payments'].round(2)}")
print(f"  Total tax collected: {pg_res['total_payment']:.2f}")

print("\nQRE + Mechanism Design complete ✓")

QUANTAL RESPONSE EQUILIBRIUM

Prisoner's Dilemma: QRE across precision λ
       λ   P(Cooperate)    Nash Pred
----------------------------------------
    0.01         0.4975  0.0 (all-D)
    0.10         0.4750  0.0 (all-D)
    0.50         0.3775  0.0 (all-D)
    1.00         0.2689  0.0 (all-D)
    2.00         0.1192  0.0 (all-D)
    5.00         0.0067  0.0 (all-D)
   10.00         0.0000  0.0 (all-D)
   50.00         0.0000  0.0 (all-D)

  λ→0: players choose randomly (≈50% cooperate)
  λ→∞: players best-respond perfectly (0% cooperate = Nash)
  QRE bridges rationality and bounded rationality ✓

Chicken Game: QRE vs Nash
  Nash mixed: P(Swerve) = 0.9
  QRE λ=  0.1: P(Swerve)=0.5795
  QRE λ=  1.0: P(Swerve)=0.2727
  QRE λ=  5.0: P(Swerve)=0.0067
  QRE λ= 20.0: P(Swerve)=0.0000



MECHANISM DESIGN: VICKREY-CLARKE-GROVES (VCG)
Objective: Design rules that make truthful reporting
each agent's dominant strategy (incentive compatible).

Test 1: VCG Auction (second-price)
  Valuations: [45, 72, 38, 91, 55]
  Winner: Agent 3 (valued at 91)
  Payment: 72.00 (second-price = 72)
  Truth-telling is dominant: bidding true value maximizes utility regardless of others ✓

Test 2: Verifying incentive compatibility
  Agent 3 true value: 91
    Reports  40: loses, utility=0.00
    Reports  60: loses, utility=0.00
    Reports  80: wins, utility=19.00
    Reports  91: wins, utility=19.00  ← TRUTH
    Reports 100: wins, utility=19.00
    Reports 120: wins, utility=19.00

  Truth-telling (report=91) achieves max utility ✓

Test 3: VCG Public Goods Mechanism
  Valuations: [15, 25, 10, 30, 20]
  Cost: 80, Total value: 100
  Decision: Provide ✓
  VCG Payments: [ 0.  5.  0. 10.  0.]
  Total tax collected: 15.00

QRE + Mechanism Design complete ✓


In [8]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Cournot Environment ───────────────────────────────────────────────────────
class CournotEnv:
    """
    Two-firm Cournot environment.
    State: [q1_last, q2_last]
    Action: quantity to produce (continuous [0, Q_MAX])
    Reward: profit = (P - c) * q_i where P = a - b*(q1+q2)
    Nash: q* = (a-c)/(3b) ≈ 30 for default params
    """
    def __init__(self, a=100, b=1, c=10, q_max=60):
        self.a = a; self.b = b; self.c = c; self.q_max = q_max
        self.q_nash = (a - c) / (3 * b)
        self.pi_nash = (a - 2*c + c) / 3 * self.q_nash  # simplification

    def profit(self, q1, q2):
        P = max(0, self.a - self.b*(q1+q2))
        pi1 = (P - self.c) * q1
        pi2 = (P - self.c) * q2
        return pi1, pi2

    def best_response(self, q_other):
        return max(0, (self.a - self.c - self.b*q_other) / (2*self.b))


# ── Gradient Ascent Multi-Agent RL ───────────────────────────────────────────
class GradientAgent:
    """
    Policy gradient agent with Gaussian policy.
    Policy: q ~ N(μ, σ²) where μ is learned.
    Update: μ ← μ + α * ∇_μ E[profit]
    """
    def __init__(self, q_init=20.0, lr=0.05, sigma=3.0, name="Agent"):
        self.mu    = q_init
        self.lr    = lr
        self.sigma = sigma
        self.name  = name
        self.history_mu = [q_init]
        self.history_q  = []
        self.history_pi = []

    def act(self):
        q = np.clip(np.random.normal(self.mu, self.sigma), 0, 60)
        self.history_q.append(q)
        return q

    def update(self, q_self, profit):
        # REINFORCE gradient: ∇_μ log π(q|μ) * profit
        # = (q - μ)/σ² * profit
        grad      = (q_self - self.mu) / (self.sigma**2) * profit
        self.mu   = np.clip(self.mu + self.lr * grad, 0, 60)
        self.sigma = max(0.5, self.sigma * 0.9995)  # decay exploration
        self.history_mu.append(self.mu)
        self.history_pi.append(profit)


# ── Fictitious Play ───────────────────────────────────────────────────────────
def fictitious_play(env, rounds=500):
    """
    Fictitious play: each agent best-responds to the
    empirical average of opponent's past actions.
    Classic convergence result for zero-sum games;
    also works for many non-zero-sum games.
    """
    q1_hist = [np.random.uniform(10, 50)]
    q2_hist = [np.random.uniform(10, 50)]

    for t in range(1, rounds):
        q1_avg = np.mean(q1_hist)
        q2_avg = np.mean(q2_hist)
        q1_new = env.best_response(q2_avg) + np.random.normal(0, 1)
        q2_new = env.best_response(q1_avg) + np.random.normal(0, 1)
        q1_hist.append(np.clip(q1_new, 0, 60))
        q2_hist.append(np.clip(q2_new, 0, 60))

    return np.array(q1_hist), np.array(q2_hist)


# ── Run experiments ───────────────────────────────────────────────────────────
np.random.seed(42)
env = CournotEnv()

print("=" * 60)
print("MULTI-AGENT RL — COURNOT COMPETITION")
print("=" * 60)
print(f"Nash Equilibrium: q₁* = q₂* = {env.q_nash:.2f}")
print(f"Monopoly quantity: {(env.a-env.c)/(2*env.b):.2f}")
print(f"Competitive quantity: {(env.a-env.c)/env.b:.2f}")

# Experiment 1: Gradient ascent agents
print("\nExperiment 1: Gradient Ascent Agents")
agent1 = GradientAgent(q_init=15.0, lr=0.08, name="Firm 1")
agent2 = GradientAgent(q_init=45.0, lr=0.08, name="Firm 2")

n_rounds = 1000
for t in range(n_rounds):
    q1 = agent1.act()
    q2 = agent2.act()
    pi1, pi2 = env.profit(q1, q2)
    agent1.update(q1, pi1)
    agent2.update(q2, pi2)

print(f"  Final Firm 1 quantity: {agent1.mu:.2f} "
      f"(Nash target: {env.q_nash:.2f})")
print(f"  Final Firm 2 quantity: {agent2.mu:.2f} "
      f"(Nash target: {env.q_nash:.2f})")
print(f"  Convergence to Nash: "
      f"{'Yes ✓' if abs(agent1.mu-env.q_nash)<5 else 'Partial'}")

# Experiment 2: Fictitious play
print("\nExperiment 2: Fictitious Play")
q1_fp, q2_fp = fictitious_play(env, rounds=500)
print(f"  Final Firm 1: {q1_fp[-1]:.2f}, "
      f"Final Firm 2: {q2_fp[-1]:.2f}")
print(f"  Nash: {env.q_nash:.2f}")
print(f"  Converged: "
      f"{'Yes ✓' if abs(q1_fp[-1]-env.q_nash)<3 else 'Partial'}")

# Experiment 3: Collusion detection
# Do agents learn to collude (produce less = monopoly)?
print("\nExperiment 3: Can Agents Learn to Collude?")
# Low learning rate + memory of past profits
agent_c1 = GradientAgent(q_init=20.0, lr=0.02, sigma=5.0, name="Colluder 1")
agent_c2 = GradientAgent(q_init=20.0, lr=0.02, sigma=5.0, name="Colluder 2")

for t in range(2000):
    q1 = agent_c1.act()
    q2 = agent_c2.act()
    pi1, pi2 = env.profit(q1, q2)
    agent_c1.update(q1, pi1)
    agent_c2.update(q2, pi2)

monopoly_q = (env.a - env.c) / (2*env.b)
print(f"  Final quantities: {agent_c1.mu:.2f}, {agent_c2.mu:.2f}")
print(f"  Monopoly optimum: {monopoly_q:.2f} each")
print(f"  Nash equilibrium: {env.q_nash:.2f} each")
print(f"  Outcome: {'Approaching collusion' if agent_c1.mu < env.q_nash else 'Nash behavior'}")

# ── Visualization ─────────────────────────────────────────────────────────────
fig_rl = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Gradient Ascent: Quantity Convergence",
        "Gradient Ascent: Profit Over Time",
        "Fictitious Play: Convergence to Nash",
        "Quantity Path in Strategy Space"
    ]
)

# Plot 1: Quantity convergence
t_ax = list(range(n_rounds+1))
fig_rl.add_trace(go.Scatter(x=t_ax, y=agent1.history_mu,
    name='Firm 1 (μ)', line=dict(color='#60a5fa',width=2)),row=1,col=1)
fig_rl.add_trace(go.Scatter(x=t_ax, y=agent2.history_mu,
    name='Firm 2 (μ)', line=dict(color='#34d399',width=2)),row=1,col=1)
fig_rl.add_hline(y=env.q_nash,line_dash="dash",line_color="#f59e0b",
                  line_width=2,row=1,col=1)
fig_rl.add_annotation(x=n_rounds*0.8,y=env.q_nash+2,
    text=f"Nash q*={env.q_nash:.1f}",
    font=dict(color="#f59e0b",size=10),showarrow=False,row=1,col=1)

# Plot 2: Rolling profit
window = 50
r1_roll = pd.Series(agent1.history_pi).rolling(window).mean()
r2_roll = pd.Series(agent2.history_pi).rolling(window).mean()
fig_rl.add_trace(go.Scatter(x=list(range(n_rounds)),y=r1_roll,
    name='Firm 1 profit',line=dict(color='#60a5fa',width=2),
    showlegend=False),row=1,col=2)
fig_rl.add_trace(go.Scatter(x=list(range(n_rounds)),y=r2_roll,
    name='Firm 2 profit',line=dict(color='#34d399',width=2),
    showlegend=False),row=1,col=2)

# Plot 3: Fictitious play
t_fp = list(range(len(q1_fp)))
fig_rl.add_trace(go.Scatter(x=t_fp,y=q1_fp,
    name='FP Firm 1',line=dict(color='#a78bfa',width=1.5),
    showlegend=False),row=2,col=1)
fig_rl.add_trace(go.Scatter(x=t_fp,y=q2_fp,
    name='FP Firm 2',line=dict(color='#fb923c',width=1.5),
    showlegend=False),row=2,col=1)
fig_rl.add_hline(y=env.q_nash,line_dash="dash",
                  line_color="#f59e0b",line_width=2,row=2,col=1)

# Plot 4: Strategy space path
fig_rl.add_trace(go.Scatter(
    x=agent1.history_mu[::10], y=agent2.history_mu[::10],
    mode='lines+markers', name='Strategy path',
    line=dict(color='#60a5fa',width=1.5),
    marker=dict(size=4,color=list(range(len(agent1.history_mu[::10]))),
                colorscale='Blues'),
    showlegend=False
),row=2,col=2)
fig_rl.add_trace(go.Scatter(
    x=[env.q_nash],y=[env.q_nash],mode='markers',name='Nash Eq',
    marker=dict(color='#f59e0b',size=14,symbol='star')
),row=2,col=2)

# Best response curves in strategy space
q_br = np.linspace(0,60,100)
br1  = [env.best_response(q) for q in q_br]
br2  = [env.best_response(q) for q in q_br]
fig_rl.add_trace(go.Scatter(x=q_br,y=br1,mode='lines',
    name='BR₁',line=dict(color='#f87171',width=1.5,dash='dot'),
    showlegend=False),row=2,col=2)
fig_rl.add_trace(go.Scatter(x=br2,y=q_br,mode='lines',
    name='BR₂',line=dict(color='#34d399',width=1.5,dash='dot'),
    showlegend=False),row=2,col=2)

fig_rl.update_layout(
    title="Multi-Agent RL: Two Firms Learning Cournot Competition",
    template="plotly_dark",height=700,
    paper_bgcolor="#0e1117",plot_bgcolor="#0e1117",
    font=dict(color='white')
)
fig_rl.update_xaxes(gridcolor='#1f2937')
fig_rl.update_yaxes(gridcolor='#1f2937')
for r,c,xl,yl in [(1,1,"Round","Quantity μ"),
                   (1,2,"Round","Avg Profit"),
                   (2,1,"Round","Quantity"),
                   (2,2,"Firm 1 Quantity","Firm 2 Quantity")]:
    fig_rl.update_xaxes(title_text=xl,row=r,col=c)
    fig_rl.update_yaxes(title_text=yl,row=r,col=c)
fig_rl.show()

print("\nMulti-agent RL complete ✓")

import pandas as pd

MULTI-AGENT RL — COURNOT COMPETITION
Nash Equilibrium: q₁* = q₂* = 30.00
Monopoly quantity: 45.00
Competitive quantity: 90.00

Experiment 1: Gradient Ascent Agents
  Final Firm 1 quantity: 60.00 (Nash target: 30.00)
  Final Firm 2 quantity: 60.00 (Nash target: 30.00)
  Convergence to Nash: Partial

Experiment 2: Fictitious Play
  Final Firm 1: 30.48, Final Firm 2: 29.69
  Nash: 30.00
  Converged: Yes ✓

Experiment 3: Can Agents Learn to Collude?
  Final quantities: 22.43, 57.44
  Monopoly optimum: 45.00 each
  Nash equilibrium: 30.00 each
  Outcome: Approaching collusion



Multi-agent RL complete ✓


In [9]:
%%writefile game_theory_dashboard.py
import streamlit as st
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.optimize import linprog, minimize
from scipy.integrate import solve_ivp
import networkx as nx
import itertools
from math import factorial
import warnings
warnings.filterwarnings('ignore')

st.set_page_config(page_title="Game Theory Simulator",
                   layout="wide", page_icon="♟️",
                   initial_sidebar_state="expanded")

st.markdown("""
<style>
  .stApp{background:#0e1117;color:#fafafa}
  [data-testid="stSidebar"]{background:#161b22}
  .metric-card{background:linear-gradient(135deg,#1f2937,#111827);
    border:1px solid #374151;border-radius:12px;
    padding:1rem;text-align:center;margin-bottom:.5rem}
  .metric-value{font-size:1.5rem;font-weight:800;color:#60a5fa}
  .metric-label{font-size:.75rem;color:#9ca3af;margin-top:.3rem}
  .sh{background:linear-gradient(90deg,#1e3a5f,#0e1117);
    border-left:4px solid #3b82f6;padding:.6rem 1rem;
    border-radius:0 8px 8px 0;margin:1rem 0 .5rem 0;
    font-size:1rem;font-weight:700;color:#93c5fd}
</style>
""", unsafe_allow_html=True)

# ── All core functions ────────────────────────────────────────────────────────
def find_pure_nash(A, B):
    eq = []
    for r in range(A.shape[0]):
        for c in range(A.shape[1]):
            if A[r,c]==np.max(A[:,c]) and B[r,c]==np.max(B[r,:]):
                eq.append((r,c,A[r,c],B[r,c]))
    return eq

def find_mixed_nash_2x2(A, B):
    res = {}
    dp = B[0,0]-B[1,0]-B[0,1]+B[1,1]
    dq = A[0,0]-A[0,1]-A[1,0]+A[1,1]
    if abs(dp)>1e-10:
        p=(B[1,1]-B[1,0])/dp
        if 0<=p<=1: res['p']=round(p,4)
    if abs(dq)>1e-10:
        q=(A[1,1]-A[0,1])/dq
        if 0<=q<=1: res['q']=round(q,4)
    return res

def find_correlated_eq(A, B):
    n,m=A.shape; nv=n*m
    c_obj=-(A+B).flatten()
    A_ub=[]; b_ub=[]
    for r in range(n):
        for r2 in range(n):
            if r==r2: continue
            row=np.zeros(nv)
            for c in range(m): row[r*m+c]=-(A[r,c]-A[r2,c])
            A_ub.append(row); b_ub.append(0)
    for c in range(m):
        for c2 in range(m):
            if c==c2: continue
            row=np.zeros(nv)
            for r in range(n): row[r*m+c]=-(B[r,c]-B[r,c2])
            A_ub.append(row); b_ub.append(0)
    try:
        res=linprog(c_obj,A_ub=np.array(A_ub),b_ub=np.array(b_ub),
                    A_eq=np.ones((1,nv)),b_eq=[1.],
                    bounds=[(0,None)]*nv,method='highs')
        if res.success:
            p=res.x.reshape(n,m)
            return p,np.sum(p*(A+B))
    except: pass
    return None,None

def replicator_rhs(t, x, A):
    f=A@x; fbar=x@f
    return x*(f-fbar)

def run_replicator(A, x0, t_max=15, n_pts=500):
    sol=solve_ivp(replicator_rhs,(0,t_max),x0,
                  t_eval=np.linspace(0,t_max,n_pts),
                  args=(A,),method='RK45',rtol=1e-8)
    return sol.t,sol.y.T

def shapley_value_fn(v, n):
    players=list(range(n)); phi=np.zeros(n)
    for i in players:
        others=[p for p in players if p!=i]
        for r in range(len(others)+1):
            for S_list in itertools.combinations(others,r):
                S=frozenset(S_list); Sw=frozenset(S_list+(i,))
                w=(factorial(len(S))*factorial(n-len(S)-1)/factorial(n))
                phi[i]+=w*(v(Sw)-v(S))
    return phi

def logit_qre_fn(A, B, lam, max_iter=500):
    n,m=A.shape
    p=np.ones(n)/n; q=np.ones(m)/m
    for _ in range(max_iter):
        eu_r=A@q; eu_c=B.T@p
        pn=np.exp(lam*eu_r); pn/=pn.sum()
        qn=np.exp(lam*eu_c); qn/=qn.sum()
        if np.max(np.abs(pn-p))<1e-10 and np.max(np.abs(qn-q))<1e-10: break
        p,q=pn,qn
    return p,q

def vcg_auction_fn(vals):
    vals=np.array(vals,dtype=float); winner=np.argmax(vals)
    alloc=np.zeros(len(vals)); alloc[winner]=1
    pay=np.zeros(len(vals))
    pay[winner]=max(v for j,v in enumerate(vals) if j!=winner)
    return winner,pay,alloc

class CournotEnv:
    def __init__(self,a=100,b=1,c=10):
        self.a=a;self.b=b;self.c=c
    def profit(self,q1,q2):
        P=max(0,self.a-self.b*(q1+q2))
        return (P-self.c)*q1,(P-self.c)*q2
    def best_response(self,q_other):
        return max(0,(self.a-self.c-self.b*q_other)/(2*self.b))
    @property
    def q_nash(self): return (self.a-self.c)/(3*self.b)

class GradientAgent:
    def __init__(self,q0=20.,lr=0.08,sig=3.):
        self.mu=float(q0);self.lr=lr;self.sigma=sig
        self.hist_mu=[float(q0)];self.hist_pi=[]
    def act(self):
        return float(np.clip(np.random.normal(self.mu,self.sigma),0,80))
    def update(self,q,pi):
        grad=(q-self.mu)/(self.sigma**2)*pi
        self.mu=float(np.clip(self.mu+self.lr*grad,0,80))
        self.sigma=max(0.5,self.sigma*0.9995)
        self.hist_mu.append(self.mu);self.hist_pi.append(pi)

COLORS=['#60a5fa','#34d399','#f59e0b','#f87171','#a78bfa','#fb923c']

# ── Sidebar ───────────────────────────────────────────────────────────────────
with st.sidebar:
    st.markdown("## ♟️ Game Theory Simulator")
    st.markdown("*8 modules · Nobel Prize mathematics*")
    st.divider()
    for m in ["Nash Equilibrium Solver","Correlated Equilibrium (Nobel 2005)",
              "Classic Games (PD, Cournot, Public Goods)",
              "Evolutionary Replicator Dynamics",
              "Shapley Values + SHAP Connection (Nobel 2012)",
              "Network Games on Graphs",
              "Quantal Response Equilibrium",
              "Mechanism Design — VCG (Nobel 2007)",
              "Multi-Agent RL (MAKOTO Extension)"]:
        st.markdown(f"· {m}")
    st.divider()
    st.markdown("**Nubaid Khan** | Game Theory Simulator")

# ── Tabs ──────────────────────────────────────────────────────────────────────
tabs=st.tabs([
    "📋 Overview",
    "🎯 Nash Solver",
    "🎮 Classic Games",
    "🧬 Evolutionary",
    "🔷 Shapley + SHAP",
    "🕸️ Network Games",
    "🎲 QRE",
    "⚙️ Mechanism Design",
    "🤖 Multi-Agent RL"
])

# ── TAB 0: Overview ───────────────────────────────────────────────────────────
with tabs[0]:
    st.markdown("# Game Theory Simulator")
    st.markdown("*From Nash equilibrium to mechanism design — computational game theory across 8 modules*")
    st.divider()

    c1,c2,c3,c4=st.columns(4)
    for col,(label,val) in zip([c1,c2,c3,c4],[
        ("Modules","8"),("Nobel Prizes Connected","4"),
        ("Equilibrium Concepts","5"),("Algorithms","7+")
    ]):
        col.markdown(f"<div class='metric-card'>"
                     f"<div class='metric-value'>{val}</div>"
                     f"<div class='metric-label'>{label}</div>"
                     f"</div>",unsafe_allow_html=True)

    st.markdown("")
    st.markdown("<div class='sh'>Module Map</div>",unsafe_allow_html=True)
    overview_df=pd.DataFrame({
        'Module':["Nash Solver","Classic Games","Evolutionary",
                  "Shapley + SHAP","Network Games","QRE",
                  "Mechanism Design","Multi-Agent RL"],
        'Nobel Connection':["Nash (1994)","Card & Krueger (2021)",
                            "Maynard Smith (not Nobel but foundational)",
                            "Shapley (2012)","Network economics (2024-adjacent)",
                            "Behavioral economics (Thaler 2017)",
                            "Hurwicz, Maskin, Myerson (2007)",
                            "MAKOTO extension"],
        'Key Concept':["Pure + Mixed + Correlated Eq.",
                       "PD, Cournot, Public Goods",
                       "Replicator dynamics ODE",
                       "Fair value division + ML explainability",
                       "Topology changes equilibrium",
                       "Bounded rationality extension of Nash",
                       "VCG + truthful mechanism",
                       "Gradient ascent + fictitious play"],
        'Innovative':["Correlated Eq. via LP ✓",
                      "Iterated tournament ✓",
                      "Simplex phase portrait ✓",
                      "SHAP bridge ✓",
                      "5 topologies compared ✓",
                      "λ-sweep visualization ✓",
                      "IC verification ✓",
                      "Collusion detection ✓"]
    })
    st.dataframe(overview_df,use_container_width=True,hide_index=True)

    st.markdown("<div class='sh'>Connection to MAKOTO</div>",
                unsafe_allow_html=True)
    st.markdown("""
MAKOTO's stated limitation (Section VI) was its **single-agent design** — one planner
optimizing global policy as if no strategic interactions existed between countries.
This simulator directly builds what MAKOTO needs next:

- **Multi-Agent RL tab**: Two PPO-style agents learning Cournot competition —
  the same framework extended to N country-agents
- **Mechanism Design tab**: VCG mechanisms that could coordinate country-level
  policy without requiring a single global planner
- **Network Games tab**: Public goods provision on graphs — models international
  cooperation where payoffs depend on which countries are allies

This is a documented research arc: MAKOTO identified the gap; this project builds the solution.
    """)

# ── TAB 1: Nash Solver ────────────────────────────────────────────────────────
with tabs[1]:
    st.markdown("# Nash Equilibrium Solver")
    st.markdown("*Pure · Mixed · Correlated — all three equilibrium concepts in one solver*")
    st.divider()

    preset=st.selectbox("Load preset:",
        ["Custom","Prisoner's Dilemma","Chicken",
         "Battle of the Sexes","Stag Hunt","Matching Pennies"])

    presets={
        "Prisoner's Dilemma":([[-1,-3],[0,-2]],[[-1,0],[-3,-2]],
            ["Cooperate","Defect"],["Cooperate","Defect"]),
        "Chicken":([[0,-1],[1,-10]],[[0,1],[-1,-10]],
            ["Swerve","Straight"],["Swerve","Straight"]),
        "Battle of the Sexes":([[3,0],[0,1]],[[1,0],[0,3]],
            ["Opera","Football"],["Opera","Football"]),
        "Stag Hunt":([[4,0],[3,3]],[[4,3],[0,3]],
            ["Stag","Hare"],["Stag","Hare"]),
        "Matching Pennies":([[1,-1],[-1,1]],[[-1,1],[1,-1]],
            ["Heads","Tails"],["Heads","Tails"]),
    }

    if preset!="Custom":
        A_d,B_d,rl,cl=presets[preset]
    else:
        A_d=[[3,0],[0,5]];B_d=[[3,5],[0,0]];rl=["Up","Down"];cl=["Left","Right"]

    c1,c2=st.columns(2)
    A_in=[]
    with c1:
        st.markdown("**Row Player (A)**")
        for i in range(len(rl)):
            row=[];cs=st.columns(len(cl))
            for j,col in enumerate(cs):
                v=col.number_input(f"A{i}{j}",value=float(A_d[i][j]),
                    key=f"A{i}{j}",label_visibility="collapsed")
                row.append(v)
            A_in.append(row)
    B_in=[]
    with c2:
        st.markdown("**Col Player (B)**")
        for i in range(len(rl)):
            row=[];cs=st.columns(len(cl))
            for j,col in enumerate(cs):
                v=col.number_input(f"B{i}{j}",value=float(B_d[i][j]),
                    key=f"B{i}{j}",label_visibility="collapsed")
                row.append(v)
            B_in.append(row)

    A=np.array(A_in,dtype=float)
    B=np.array(B_in,dtype=float)

    if st.button("Solve",type="primary"):
        pure=find_pure_nash(A,B)
        mixed=find_mixed_nash_2x2(A,B)
        ce,sw=find_correlated_eq(A,B)

        c1,c2,c3=st.columns(3)
        c1.metric("Pure Nash",str(len(pure)))
        c2.metric("Mixed Nash","Exists" if mixed else "N/A")
        c3.metric("Correlated SW",f"{sw:.3f}" if sw else "N/A")

        st.markdown("<div class='sh'>Pure Strategy Nash</div>",
                    unsafe_allow_html=True)
        if pure:
            for r,c,ar,bc in pure:
                st.success(f"**({rl[r]}, {cl[c]})** → Row={ar}, Col={bc}")
        else:
            st.warning("No pure strategy Nash — mixed equilibrium only")

        if mixed:
            st.markdown("<div class='sh'>Mixed Strategy Nash</div>",
                        unsafe_allow_html=True)
            if 'p' in mixed:
                st.write(f"Row: **{mixed['p']}** on {rl[0]}, "
                         f"**{round(1-mixed['p'],4)}** on {rl[1]}")
            if 'q' in mixed:
                st.write(f"Col: **{mixed['q']}** on {cl[0]}, "
                         f"**{round(1-mixed['q'],4)}** on {cl[1]}")

        if ce is not None:
            st.markdown("<div class='sh'>Correlated Equilibrium "
                        "(Aumann 2005 Nobel)</div>",
                        unsafe_allow_html=True)
            st.dataframe(pd.DataFrame(ce.round(4),index=rl,columns=cl),
                         use_container_width=True)
            nash_sw=max(A[r,c]+B[r,c] for r,c,_,_ in pure) if pure else 0
            if sw>nash_sw+0.001:
                st.success(f"Correlation improves social welfare by "
                           f"{sw-nash_sw:.4f} over best Nash ✓")
            else:
                st.info("Correlated equilibrium matches Nash welfare here")

# ── TAB 2: Classic Games ──────────────────────────────────────────────────────
with tabs[2]:
    st.markdown("# Classic Games")
    st.divider()

    gtab=st.radio("",["Prisoner's Dilemma","Cournot Oligopoly",
                       "Public Goods"],horizontal=True)

    if gtab=="Prisoner's Dilemma":
        st.markdown("### Iterated Prisoner's Dilemma Tournament")
        STRATS={
            'Always Defect':    lambda h:'D',
            'Always Cooperate': lambda h:'C',
            'Tit-for-Tat':      lambda h:h[-1] if h else 'C',
            'Grim Trigger':     lambda h:'D' if 'D' in h else 'C',
            'Random':           lambda h:np.random.choice(['C','D']),
        }
        c1,c2=st.columns(2)
        s1n=c1.selectbox("Player 1",list(STRATS.keys()),index=2)
        s2n=c2.selectbox("Player 2",list(STRATS.keys()),index=0)
        rds=st.slider("Rounds",10,200,60)

        PAYOFF={('C','C'):(-1,-1),('C','D'):(-3,0),
                ('D','C'):(0,-3),('D','D'):(-2,-2)}
        s1=STRATS[s1n];s2=STRATS[s2n]
        h1,h2,sc1,sc2=[],[],[],[]
        np.random.seed(42)
        for _ in range(rds):
            a1=s1(h2);a2=s2(h1)
            p1,p2=PAYOFF[(a1,a2)]
            h1.append(a1);h2.append(a2);sc1.append(p1);sc2.append(p2)

        fig=go.Figure()
        fig.add_trace(go.Scatter(y=np.cumsum(sc1),mode='lines',
            name=s1n,line=dict(color='#60a5fa',width=2.5)))
        fig.add_trace(go.Scatter(y=np.cumsum(sc2),mode='lines',
            name=s2n,line=dict(color='#34d399',width=2.5)))
        fig.update_layout(xaxis_title="Round",yaxis_title="Cumulative Score",
            template="plotly_dark",height=380,
            paper_bgcolor="#0e1117",plot_bgcolor="#0e1117",font=dict(color='white'))
        st.plotly_chart(fig,use_container_width=True)

        coop1=h1.count('C')/rds; coop2=h2.count('C')/rds
        c1,c2,c3=st.columns(3)
        c1.metric(f"{s1n} cooperation rate",f"{coop1*100:.1f}%")
        c2.metric(f"{s2n} cooperation rate",f"{coop2*100:.1f}%")
        c3.metric("Scores",f"{sum(sc1)} vs {sum(sc2)}")

    elif gtab=="Cournot Oligopoly":
        st.markdown("### Cournot Duopoly — Best Response Curves")
        c1,c2,c3=st.columns(3)
        a=c1.slider("Demand (a)",50,200,100)
        b=c2.slider("Slope (b)",1,5,1)
        cost=c3.slider("Cost (c)",0,50,10)

        env=CournotEnv(a=a,b=b,c=cost)
        qn=env.q_nash
        Pn=a-b*2*qn; pin=(Pn-cost)*qn

        c1,c2,c3,c4=st.columns(4)
        c1.metric("Nash q*",f"{qn:.2f}")
        c2.metric("Price P*",f"{Pn:.2f}")
        c3.metric("Profit π*",f"{pin:.2f}")
        c4.metric("vs Monopoly",f"{qn/((a-cost)/(2*b))*100:.0f}% of Q_m")

        q_range=np.linspace(0,min(a/b,80),200)
        br=[env.best_response(q) for q in q_range]
        fig2=go.Figure()
        fig2.add_trace(go.Scatter(x=q_range,y=br,mode='lines',
            name='BR₁(q₂)',line=dict(color='#60a5fa',width=2.5)))
        fig2.add_trace(go.Scatter(x=br,y=q_range,mode='lines',
            name='BR₂(q₁)',line=dict(color='#34d399',width=2.5)))
        fig2.add_trace(go.Scatter(x=[qn],y=[qn],mode='markers',
            name='Nash Eq',
            marker=dict(color='#f59e0b',size=14,symbol='star')))
        fig2.add_trace(go.Scatter(
            x=[(a-cost)/(2*b)],y=[0],mode='markers',name='Monopoly',
            marker=dict(color='#f87171',size=10,symbol='diamond')))
        fig2.update_layout(xaxis_title="q₁",yaxis_title="q₂",
            template="plotly_dark",height=420,
            paper_bgcolor="#0e1117",plot_bgcolor="#0e1117",font=dict(color='white'))
        st.plotly_chart(fig2,use_container_width=True)

    else:
        st.markdown("### Public Goods Game — Free Rider Problem")
        n_p=st.slider("Players",3,15,6)
        ret=st.slider("MPCR (return rate)",1.0,3.0,1.6,0.1)
        rds2=st.slider("Rounds",10,60,30)

        contribs=np.ones(n_p)*5.0
        hist_c,hist_p=[],[]
        np.random.seed(42)
        for _ in range(rds2):
            total=np.sum(contribs); share=ret*total/n_p
            payoff=10-contribs+share
            hist_c.append(np.mean(contribs))
            hist_p.append(np.mean(payoff))
            best=np.argmax(payoff); nc=contribs.copy()
            for i in range(n_p):
                if payoff[i]<np.mean(payoff):
                    nc[i]=contribs[best]*.7+contribs[i]*.3+np.random.normal(0,.3)
                nc[i]=np.clip(nc[i],0,10)
            contribs=nc

        fig3=make_subplots(rows=1,cols=2,
            subplot_titles=["Avg Contribution","Avg Payoff"])
        fig3.add_trace(go.Scatter(y=hist_c,mode='lines',
            line=dict(color='#60a5fa',width=2.5),showlegend=False),row=1,col=1)
        fig3.add_hline(y=0,line_dash="dot",line_color="#f87171",
                       annotation_text="Nash (0)",row=1,col=1)
        fig3.add_hline(y=10,line_dash="dot",line_color="#34d399",
                       annotation_text="Optimum (10)",row=1,col=1)
        fig3.add_trace(go.Scatter(y=hist_p,mode='lines',
            line=dict(color='#34d399',width=2.5),showlegend=False),row=1,col=2)
        fig3.update_layout(template="plotly_dark",height=380,
            paper_bgcolor="#0e1117",plot_bgcolor="#0e1117",font=dict(color='white'))
        st.plotly_chart(fig3,use_container_width=True)
        st.info(f"MPCR={ret:.1f}. Nash prediction: zero contribution. "
                f"{'With learning, cooperation partially emerges.' if ret>1 else 'No incentive to cooperate.'}")

# ── TAB 3: Evolutionary ───────────────────────────────────────────────────────
with tabs[3]:
    st.markdown("# Evolutionary Game Theory")
    st.markdown("*Replicator dynamics: strategies evolve by selection pressure, not rationality*")
    st.divider()

    EVO={
        "Prisoner's Dilemma":(np.array([[-1.,-3.],[0.,-2.]]),
            ['Cooperate','Defect'],"All-Defect is ESS"),
        "Hawk-Dove":(np.array([[0.,3.],[1.,2.]]),
            ['Hawk','Dove'],"Mixed equilibrium — both coexist"),
        "Stag Hunt":(np.array([[4.,0.],[3.,3.]]),
            ['Stag','Hare'],"Two stable equilibria"),
        "Rock-Paper-Scissors":(np.array([[0.,-1.,1.],[1.,0.,-1.],[-1.,1.,0.]]),
            ['Rock','Paper','Scissors'],"Neutrally stable cycles"),
    }

    gevo=st.selectbox("Game:",list(EVO.keys()))
    A_evo,strats,pred=EVO[gevo]
    n_s=len(strats)
    st.info(f"**Prediction:** {pred}")

    c1,c2=st.columns(2)
    x0=[]
    for i,s in enumerate(strats):
        v=c1.slider(f"Initial {s}",0.0,1.0,1.0/n_s,0.05,key=f"e{i}")
        x0.append(v)
    x0=np.array(x0,dtype=float)
    x0=x0/x0.sum() if x0.sum()>0 else np.ones(n_s)/n_s
    t_max=c2.slider("Time",5,40,15)

    t,X=run_replicator(A_evo,x0,t_max=t_max)

    fig_evo=go.Figure()
    for i,s in enumerate(strats):
        fig_evo.add_trace(go.Scatter(x=t,y=X[:,i],mode='lines',
            name=s,line=dict(color=COLORS[i],width=2.5)))
    fig_evo.update_layout(xaxis_title="Time",yaxis_title="Population Share",
        template="plotly_dark",height=420,
        paper_bgcolor="#0e1117",plot_bgcolor="#0e1117",font=dict(color='white'),
        legend=dict(orientation="h",y=1.1))
    st.plotly_chart(fig_evo,use_container_width=True)

    if n_s==3:
        st.markdown("<div class='sh'>Simplex Phase Portrait</div>",
                    unsafe_allow_html=True)
        def b2c(x):
            v=np.array([[0.,0.],[1.,0.],[.5,np.sqrt(3)/2]])
            return x[0]*v[0]+x[1]*v[1]+x[2]*v[2]

        fig_s=go.Figure()
        corners=np.array([[1,0,0],[0,1,0],[0,0,1],[1,0,0]],dtype=float)
        cart=np.array([b2c(c) for c in corners])
        fig_s.add_trace(go.Scatter(x=cart[:,0],y=cart[:,1],mode='lines',
            line=dict(color='#374151',width=2),showlegend=False))
        for i,(s,corner) in enumerate(zip(strats,[[1,0,0],[0,1,0],[0,0,1]])):
            pos=b2c(np.array(corner,dtype=float))
            fig_s.add_annotation(x=pos[0],y=pos[1]+.05,text=s,
                font=dict(color=COLORS[i],size=13),showarrow=False)
        np.random.seed(42)
        for _ in range(20):
            raw=np.random.dirichlet(np.ones(3))
            t2,X2=run_replicator(A_evo,raw,t_max=10,n_pts=200)
            ct=np.array([b2c(X2[i]) for i in range(len(t2))])
            fig_s.add_trace(go.Scatter(x=ct[:,0],y=ct[:,1],mode='lines',
                line=dict(color='rgba(96,165,250,0.35)',width=1.2),showlegend=False))
        ctr=b2c(np.array([1/3,1/3,1/3]))
        fig_s.add_trace(go.Scatter(x=[ctr[0]],y=[ctr[1]],mode='markers',
            name='Nash (1/3,1/3,1/3)',
            marker=dict(color='#f59e0b',size=12,symbol='star')))
        fig_s.update_layout(template="plotly_dark",height=480,
            paper_bgcolor="#0e1117",plot_bgcolor="#0e1117",font=dict(color='white'),
            xaxis=dict(showgrid=False,zeroline=False,showticklabels=False),
            yaxis=dict(showgrid=False,zeroline=False,showticklabels=False,
                       scaleanchor="x"))
        st.plotly_chart(fig_s,use_container_width=True)

# ── TAB 4: Shapley + SHAP ─────────────────────────────────────────────────────
with tabs[4]:
    st.markdown("# Shapley Values + SHAP Connection")
    st.markdown("*Fair value division in cooperative games — and why it's the foundation of ML explainability*")
    st.divider()

    c1,c2,c3=st.columns(3)
    v01=c1.slider("v({0,1})",0,100,50)
    v02=c2.slider("v({0,2})",0,100,30)
    v12=c3.slider("v({1,2})",0,100,20)
    v012=st.slider("Grand coalition v({0,1,2})",0,200,100)

    def cg(S):
        s=set(S)
        if s=={0,1,2}: return float(v012)
        if s=={0,1}: return float(v01)
        if s=={0,2}: return float(v02)
        if s=={1,2}: return float(v12)
        if s=={0}: return 10.
        if s=={1}: return 5.
        if s=={2}: return 8.
        return 0.

    phi=shapley_value_fn(cg,3)
    pnames=['Programmer','Designer','Manager']

    c1,c2,c3=st.columns(3)
    for col,name,p in zip([c1,c2,c3],pnames,phi):
        col.markdown(f"<div class='metric-card'>"
                     f"<div class='metric-value'>{p:.2f}</div>"
                     f"<div class='metric-label'>φ({name})</div>"
                     f"</div>",unsafe_allow_html=True)

    fig_phi=go.Figure(go.Bar(x=pnames,y=phi,
        marker_color=['#60a5fa','#34d399','#f59e0b'],
        text=[f"φ={p:.2f} ({p/max(sum(phi),1)*100:.1f}%)" for p in phi],
        textposition='outside'))
    fig_phi.add_hline(y=0,line_color="#6b7280",line_width=1)
    fig_phi.update_layout(
        title=f"Shapley Values — Grand Coalition = {v012}",
        yaxis_title="φᵢ",template="plotly_dark",height=380,
        paper_bgcolor="#0e1117",plot_bgcolor="#0e1117",font=dict(color='white'))
    st.plotly_chart(fig_phi,use_container_width=True)
    st.caption(f"Sum = {sum(phi):.2f} = {v012} ✓ efficiency axiom | "
               f"All four Shapley axioms satisfied: efficiency, symmetry, dummy, additivity")

    st.markdown("<div class='sh'>SHAP Bridge: Game Theory → Machine Learning</div>",
                unsafe_allow_html=True)
    col1,col2=st.columns(2)
    with col1:
        st.markdown("""
**In cooperative game theory:**
- Players decide whether to join coalitions
- Value function v(S) = output of coalition S
- Shapley value = fair share of grand coalition value

**In machine learning (SHAP):**
- Features are the "players"
- Coalition = subset of features revealed to model
- Value function = model's expected prediction given those features
- SHAP value = each feature's fair attribution to the prediction

**They are mathematically identical.** SHAP values satisfy the
four Shapley axioms not by design coincidence but because
Lundberg & Lee (2017) explicitly derived them from cooperative
game theory. A 1953 economics result became the 2023 gold
standard for AI explainability.
        """)
    with col2:
        st.markdown("""
**Connection to MAKOTO:**

In MAKOTO, permutation-based feature importance measures
how much each macroeconomic predictor contributes to
forecast accuracy. This is a Shapley-adjacent method —
approximating the Shapley value by sampling permutations
rather than computing it exactly.

The top MAKOTO predictor (lag_capital_form, ΔMAE=+0.0433)
can be interpreted as having the highest Shapley value in
the cooperative forecasting game where features are players
and the grand coalition's value is forecast accuracy.

Building the exact Shapley value calculator here makes that
connection explicit and rigorous.
        """)

# ── TAB 5: Network Games ──────────────────────────────────────────────────────
with tabs[5]:
    st.markdown("# Network Games")
    st.markdown("*Graph topology changes equilibrium — the same game produces different outcomes on different networks*")
    st.divider()

    c1,c2,c3=st.columns(3)
    net=c1.selectbox("Topology",["Complete","Star","Random","Scale-Free","Small World"])
    n_nd=c2.slider("Nodes",6,20,10)
    ret_n=c3.slider("Return rate",1.0,3.0,1.6,0.1)

    seed=42
    if net=="Complete":    G=nx.complete_graph(n_nd)
    elif net=="Star":      G=nx.star_graph(n_nd-1)
    elif net=="Random":    G=nx.erdos_renyi_graph(n_nd,.4,seed=seed)
    elif net=="Scale-Free":G=nx.barabasi_albert_graph(n_nd,2,seed=seed)
    else:                  G=nx.watts_strogatz_graph(n_nd,4,.3,seed=seed)

    pos=nx.spring_layout(G,seed=seed)
    np.random.seed(42)
    cs=np.random.uniform(3,7,n_nd)
    for _ in range(30):
        pf=np.zeros(n_nd)
        for i in G.nodes:
            nb=list(G.neighbors(i))
            if not nb: pf[i]=10-cs[i]; continue
            pool=cs[i]+sum(cs[j] for j in nb)
            pf[i]=10-cs[i]+ret_n*pool/(len(nb)+1)
        nc=cs.copy()
        for i in G.nodes:
            nb=list(G.neighbors(i))
            if nb:
                bst=nb[np.argmax([pf[j] for j in nb])]
                if pf[bst]>pf[i]:
                    nc[i]=cs[bst]*.7+cs[i]*.3+np.random.normal(0,.3)
            nc[i]=np.clip(nc[i],0,10)
        cs=nc

    ex,ey=[],[]
    for e in G.edges():
        x0,y0=pos[e[0]];x1,y1=pos[e[1]]
        ex+=[x0,x1,None];ey+=[y0,y1,None]
    nxa=[pos[n][0] for n in G.nodes]
    nya=[pos[n][1] for n in G.nodes]

    fig_net=go.Figure()
    fig_net.add_trace(go.Scatter(x=ex,y=ey,mode='lines',
        line=dict(color='rgba(148,163,184,0.3)',width=1.5),showlegend=False))
    fig_net.add_trace(go.Scatter(x=nxa,y=nya,mode='markers+text',
        marker=dict(size=28,color=cs,colorscale='RdYlGn',
                    colorbar=dict(title='Contribution'),cmin=0,cmax=10,
                    line=dict(color='white',width=1.5)),
        text=[f"{c:.1f}" for c in cs],
        textfont=dict(color='black',size=9),showlegend=False))
    fig_net.update_layout(
        title=f"{net} Network — Final Contributions | Density={nx.density(G):.3f}",
        template="plotly_dark",height=460,
        paper_bgcolor="#0e1117",plot_bgcolor="#0e1117",font=dict(color='white'),
        xaxis=dict(showgrid=False,zeroline=False,showticklabels=False),
        yaxis=dict(showgrid=False,zeroline=False,showticklabels=False))
    st.plotly_chart(fig_net,use_container_width=True)

    c1,c2,c3,c4=st.columns(4)
    c1.metric("Avg Contribution",f"{np.mean(cs):.2f}")
    c2.metric("Std Dev",f"{np.std(cs):.2f}")
    c3.metric("Density",f"{nx.density(G):.3f}")
    c4.metric("Avg Clustering",f"{nx.average_clustering(G):.3f}")

# ── TAB 6: QRE ───────────────────────────────────────────────────────────────
with tabs[6]:
    st.markdown("# Quantal Response Equilibrium")
    st.markdown("*Bounded rationality: players make mistakes, but better responses are chosen more often*")
    st.divider()

    c1,c2=st.columns(2)
    gqre=c1.selectbox("Game:",
        ["Prisoner's Dilemma","Chicken","Battle of the Sexes"])
    lam=c2.slider("Precision λ",0.01,20.0,1.0,0.01)

    qre_map={
        "Prisoner's Dilemma":(
            np.array([[-1.,-3.],[0.,-2.]]),
            np.array([[-1.,0.],[-3.,-2.]]),["Cooperate","Defect"]),
        "Chicken":(
            np.array([[0.,-1.],[1.,-10.]]),
            np.array([[0.,1.],[-1.,-10.]]),["Swerve","Straight"]),
        "Battle of the Sexes":(
            np.array([[3.,0.],[0.,1.]]),
            np.array([[1.,0.],[0.,3.]]),["Opera","Football"]),
    }
    Aq,Bq,lbq=qre_map[gqre]
    pq,_=logit_qre_fn(Aq,Bq,lam)
    pn=find_mixed_nash_2x2(Aq,Bq)

    c1,c2,c3=st.columns(3)
    c1.metric(f"QRE P({lbq[0]})",f"{pq[0]:.4f}",f"λ={lam:.2f}")
    c2.metric(f"Nash P({lbq[0]})",f"{pn.get('p','all-pure')}")
    c3.metric("Rationality",
              "Near-Nash" if lam>5 else "Bounded rational" if lam>0.5 else "Near-random")

    lams=np.logspace(-2,np.log10(20),100)
    psweep=[logit_qre_fn(Aq,Bq,l)[0][0] for l in lams]
    fig_qr=go.Figure()
    fig_qr.add_trace(go.Scatter(x=lams,y=psweep,mode='lines',
        name='QRE',line=dict(color='#60a5fa',width=2.5)))
    if 'p' in pn:
        fig_qr.add_hline(y=pn['p'],line_dash="dash",line_color="#f59e0b",
                          line_width=2,annotation_text=f"Nash={pn['p']}")
    fig_qr.add_vline(x=lam,line_dash="dot",line_color="#34d399",line_width=1.5,
                     annotation_text=f"Current λ={lam:.2f}")
    fig_qr.update_layout(
        title=f"QRE: P({lbq[0]}) vs Precision λ",
        xaxis_title="λ (log scale)",xaxis_type="log",
        yaxis_title=f"P({lbq[0]})",template="plotly_dark",height=400,
        paper_bgcolor="#0e1117",plot_bgcolor="#0e1117",font=dict(color='white'))
    st.plotly_chart(fig_qr,use_container_width=True)
    st.info("As λ→∞ players become perfectly rational and QRE converges to Nash. "
            "QRE consistently outperforms Nash in predicting actual human behavior "
            "in experimental games.")

# ── TAB 7: Mechanism Design ───────────────────────────────────────────────────
with tabs[7]:
    st.markdown("# Mechanism Design")
    st.markdown("*Design the rules that make truth-telling dominant — Nobel 2007 (Hurwicz, Maskin, Myerson)*")
    st.divider()

    mech=st.radio("",["VCG Auction","Public Goods VCG"],horizontal=True)

    if mech=="VCG Auction":
        st.markdown("### Vickrey-Clarke-Groves Second-Price Auction")
        n_b=st.slider("Bidders",2,8,5)
        cols=st.columns(n_b)
        dvals=[45,72,38,91,55,30,88,60]
        vals=[col.number_input(f"Bidder {i}",0,200,dvals[i],
              key=f"vcg{i}") for i,col in enumerate(cols)]

        w,pay,alloc=vcg_auction_fn(vals)
        c1,c2,c3=st.columns(3)
        c1.metric("Winner",f"Bidder {w}")
        c2.metric("VCG Payment",f"{pay[w]:.2f}")
        c3.metric("Winner's Value",f"{vals[w]}")
        st.success(f"Bidder {w} wins (value={vals[w]}), pays "
                   f"{pay[w]:.0f} (second-highest = {sorted(vals)[-2]}). "
                   f"Surplus = {vals[w]-pay[w]:.0f}")

        st.markdown("<div class='sh'>Incentive Compatibility Proof</div>",
                    unsafe_allow_html=True)
        true_v=vals[w]; ic=[]
        for rep in range(0,int(max(vals))+25,15):
            fv=list(vals); fv[w]=rep
            ww,pp,aa=vcg_auction_fn(fv)
            util=true_v*aa[w]-pp[w]
            ic.append({'Reported':rep,'Wins':bool(aa[w]),
                       'Pays':round(pp[w],1),'Utility':round(util,1),
                       'Truth?':'← MAX ✓' if rep==true_v else ''})
        st.dataframe(pd.DataFrame(ic),use_container_width=True,hide_index=True)
        st.caption("Utility is maximized at truthful report. VCG makes honesty a dominant strategy.")

    else:
        st.markdown("### Public Goods VCG — Clarke Tax Mechanism")
        n_ag=st.slider("Agents",3,8,5)
        cost=st.slider("Public good cost",20,200,80)
        cols=st.columns(n_ag)
        dv=[15,25,10,30,20,18,22,12]
        pgv=[col.number_input(f"Agent {i}",0,60,dv[i],key=f"pgv{i}")
             for i,col in enumerate(cols)]

        total=sum(pgv); provide=total>=cost
        c1,c2,c3=st.columns(3)
        c1.metric("Total Value",str(total))
        c2.metric("Cost",str(cost))
        c3.metric("Decision","Provide ✓" if provide else "Do Not Provide")

        if provide:
            st.success(f"Efficient: total value {total} ≥ cost {cost}. "
                       f"Net social gain: {total-cost}")
        else:
            st.error(f"Inefficient to provide: {total} < {cost}")

        pivotal=[i for i in range(n_ag)
                 if (sum(v for j,v in enumerate(pgv) if j!=i)>=cost)!=provide]
        if pivotal:
            st.info(f"Pivotal agents (pay Clarke tax): {pivotal}")
        else:
            st.info("No agent is pivotal — no Clarke tax collected")

# ── TAB 8: Multi-Agent RL ─────────────────────────────────────────────────────
with tabs[8]:
    st.markdown("# Multi-Agent RL — MAKOTO Extension")
    st.markdown("*Two firms learn Cournot competition via gradient ascent — Nash, collusion, or something else?*")
    st.divider()

    c1,c2,c3=st.columns(3)
    a_e=c1.slider("Demand (a)",50,150,100)
    c_e=c2.slider("Cost (c)",0,40,10)
    rds_rl=c3.slider("Rounds",200,2000,600)
    lr_e=st.slider("Learning rate",0.01,0.20,0.08,0.01)

    env_e=CournotEnv(a=a_e,b=1,c=c_e)
    qn_e=env_e.q_nash
    qm_e=(a_e-c_e)/2

    if st.button("Run Training",type="primary"):
        np.random.seed(42)
        ag1=GradientAgent(q0=qn_e*0.5,lr=lr_e,sig=4.)
        ag2=GradientAgent(q0=qn_e*1.5,lr=lr_e,sig=4.)

        with st.spinner(f"Training {rds_rl} rounds..."):
            for t in range(rds_rl):
                q1=ag1.act();q2=ag2.act()
                p1,p2=env_e.profit(q1,q2)
                ag1.update(q1,p1);ag2.update(q2,p2)

        avg_q=(ag1.mu+ag2.mu)/2
        gap_nash=abs(avg_q-qn_e)/qn_e*100
        gap_mono=abs(avg_q-qm_e/2)/(qm_e/2)*100

        c1,c2,c3,c4=st.columns(4)
        c1.metric("Firm 1 final q",f"{ag1.mu:.2f}")
        c2.metric("Firm 2 final q",f"{ag2.mu:.2f}")
        c3.metric("Nash target",f"{qn_e:.2f}")
        outcome=("Collusion detected 🚨" if avg_q<qn_e*0.85
                 else "Nash convergence ✓" if gap_nash<15
                 else "Learning in progress")
        c4.metric("Outcome",outcome,f"{gap_nash:.1f}% from Nash")

        fig_ml=make_subplots(rows=1,cols=2,
            subplot_titles=["Quantity Convergence","Strategy Space"])

        t_ax=list(range(len(ag1.hist_mu)))
        fig_ml.add_trace(go.Scatter(x=t_ax,y=ag1.hist_mu,
            name='Firm 1',line=dict(color='#60a5fa',width=2)),row=1,col=1)
        fig_ml.add_trace(go.Scatter(x=t_ax,y=ag2.hist_mu,
            name='Firm 2',line=dict(color='#34d399',width=2)),row=1,col=1)
        fig_ml.add_hline(y=qn_e,line_dash="dash",line_color="#f59e0b",
                          line_width=2,annotation_text=f"Nash q*={qn_e:.1f}",
                          row=1,col=1)
        fig_ml.add_hline(y=qm_e/2,line_dash="dot",line_color="#f87171",
                          line_width=1,annotation_text=f"Collude q={qm_e/2:.1f}",
                          row=1,col=1)

        step=max(1,rds_rl//80)
        fig_ml.add_trace(go.Scatter(
            x=ag1.hist_mu[::step],y=ag2.hist_mu[::step],
            mode='lines+markers',name='Path',showlegend=False,
            line=dict(color='#a78bfa',width=1.5),
            marker=dict(size=4,color=list(range(len(ag1.hist_mu[::step]))),
                        colorscale='Viridis')),row=1,col=2)
        fig_ml.add_trace(go.Scatter(x=[qn_e],y=[qn_e],mode='markers',
            name='Nash',marker=dict(color='#f59e0b',size=14,symbol='star')),
            row=1,col=2)
        q_br=np.linspace(0,min(a_e,80),100)
        br=[env_e.best_response(q) for q in q_br]
        fig_ml.add_trace(go.Scatter(x=q_br,y=br,mode='lines',showlegend=False,
            line=dict(color='#f87171',width=1,dash='dot'),name='BR₁'),
            row=1,col=2)
        fig_ml.add_trace(go.Scatter(x=br,y=q_br,mode='lines',showlegend=False,
            line=dict(color='#34d399',width=1,dash='dot'),name='BR₂'),
            row=1,col=2)

        fig_ml.update_layout(template="plotly_dark",height=480,
            paper_bgcolor="#0e1117",plot_bgcolor="#0e1117",font=dict(color='white'))
        fig_ml.update_xaxes(gridcolor='#1f2937')
        fig_ml.update_yaxes(gridcolor='#1f2937')
        fig_ml.update_xaxes(title_text="Round",row=1,col=1)
        fig_ml.update_xaxes(title_text="Firm 1 q",row=1,col=2)
        fig_ml.update_yaxes(title_text="Quantity",row=1,col=1)
        fig_ml.update_yaxes(title_text="Firm 2 q",row=1,col=2)
        st.plotly_chart(fig_ml,use_container_width=True)

        st.markdown("<div class='sh'>MAKOTO Connection</div>",
                    unsafe_allow_html=True)
        st.markdown(f"""
MAKOTO (Section VI) identified **single-agent design** as its primary limitation.
This tab is the direct extension: instead of one planner optimizing global policy,
two agents with competing interests learn simultaneously.

With learning rate **{lr_e}** and **{rds_rl}** rounds:
- Average quantity: **{avg_q:.2f}** vs Nash **{qn_e:.2f}**
- Gap from Nash: **{gap_nash:.1f}%**
- Outcome: **{outcome}**

If agents learn to collude (produce below Nash), this demonstrates that
multi-agent RL systems can develop implicit coordination without communication —
a finding with direct policy implications for antitrust and international economics.
        """)
    else:
        st.info("Set parameters above and click **Run Training** to simulate "
                "two firms learning the Cournot game.")
        c1,c2=st.columns(2)
        c1.metric("Nash equilibrium target",f"{qn_e:.2f} per firm")
        c2.metric("Monopoly (collusion) target",f"{qm_e/2:.2f} per firm")

Writing game_theory_dashboard.py


In [10]:
!pip install streamlit -q
import os, time, subprocess

os.system("pkill -f streamlit 2>/dev/null")
time.sleep(2)

subprocess.Popen(
    ["streamlit","run","game_theory_dashboard.py",
     "--server.port","8501",
     "--server.headless","true",
     "--server.enableCORS","false",
     "--server.enableXsrfProtection","false"],
    stdout=open("/tmp/st_gt.log","w"),
    stderr=subprocess.STDOUT
)

print("Starting...")
for i in range(15):
    time.sleep(1)
    try:
        import urllib.request
        urllib.request.urlopen("http://localhost:8501",timeout=1)
        print(f"Ready after {i+1}s")
        break
    except:
        print(f"Waiting... {i+1}s")

from google.colab.output import serve_kernel_port_as_window
serve_kernel_port_as_window(8501)

Starting...
Waiting... 1s
Waiting... 2s
Waiting... 3s
Waiting... 4s
Ready after 5s
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>